In [1]:
!pip -q install requests beautifulsoup4 tqdm pyreadstat


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 25.0 MB/s eta 0:00:00


In [2]:
import os
import re
import zipfile
import unicodedata
import requests
import pandas as pd
import pyreadstat
from bs4 import BeautifulSoup
from tqdm import tqdm
from pathlib import Path

# ==============================================================================
# 1. CONFIGURACIÓN PRINCIPAL
# ==============================================================================
ANIO_ENCUESTA = 2023
BASE_URL = f"https://ensanut.insp.mx/encuestas/ensanutcontinua{ANIO_ENCUESTA}/descargas.php"
CARPETA_SALIDA = f"ENSANUT_{ANIO_ENCUESTA}_Completa"
UA = "Mozilla/5.0 (compatible; ensanut-downloader-bot/5.0)"

# ==============================================================================
# 2. UTILIDADES DE TEXTO (ORTOGRAFÍA ESTRICTA)
# ==============================================================================
def restaurar_ortografia(texto):
    """
    Restaura Ñ y acentos respetando RIGUROSAMENTE mayúsculas y minúsculas.
    """
    if not isinstance(texto, str):
        return texto
    texto = texto.strip()

    reemplazos = {
        r'\bANOS\b': 'AÑOS', r'\bNINOS\b': 'NIÑOS', r'\bNUTRICION\b': 'NUTRICIÓN',
        r'\bEVALUACION\b': 'EVALUACIÓN', r'\bOBTENCION\b': 'OBTENCIÓN', r'\bFISICA\b': 'FÍSICA',
        r'\banos\b': 'años', r'\bninos\b': 'niños', r'\bnutricion\b': 'nutrición',
        r'\bevaluacion\b': 'evaluación', r'\bfisica\b': 'física',
        r'\bNinos\b': 'Niños', r'\bAnos\b': 'Años'
    }
    for patron, rep in reemplazos.items():
        texto = re.sub(patron, rep, texto)
    return texto

def limpiar_nombre_sistema(texto):
    if not texto:
        return ""
    texto = re.sub(r'[\\/*?:"<>|]', "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    texto_norm = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )
    return restaurar_ortografia(texto_norm)

def detectar_modulo(ruta_sav, base_dir):
    """
    Devuelve 'NUTRICIÓN' o 'SALUD' según la carpeta en la ruta.
    Si no lo encuentra, devuelve 'OTRO'.
    """
    base_dir = Path(base_dir).resolve()
    ruta = Path(ruta_sav).resolve()

    # Intento 1: tomar el primer nivel debajo de base_dir (lo más robusto)
    try:
        primer_nivel = ruta.relative_to(base_dir).parts[0]
        txt = primer_nivel
    except Exception:
        # Fallback: buscar en toda la ruta
        txt = " ".join(ruta.parts)

    # Normalizar (quitar acentos) y poner en mayúsculas
    txt_norm = "".join(
        c for c in unicodedata.normalize("NFD", str(txt))
        if unicodedata.category(c) != "Mn"
    ).upper()

    if "NUTRICION" in txt_norm:
        return "NUTRICIÓN"
    if "SALUD" in txt_norm:
        return "SALUD"
    return "OTRO"

# ==============================================================================
# 3. CLASE DE DESCARGA
# ==============================================================================
class EnsanutDownloader:
    def __init__(self, base_url, output_dir):
        self.base_url = base_url
        self.output_dir = output_dir
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": UA})

    def _get_indent_level(self, td_tag):
        p_tag = td_tag.find("p")
        if not p_tag:
            return 0
        style = p_tag.get("style", "")
        m = re.search(r"padding-left\s*:\s*(\d+)", style)
        return int(int(m.group(1)) / 20) if m else 0

    def get_tasks(self):
        print(f"📡 Conectando a {self.base_url}...")
        resp = self.session.get(self.base_url, timeout=60)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")

        form = soup.find("form", attrs={"method": re.compile("post", re.I)})
        if not form:
            raise RuntimeError("Error: No se encontró formulario web.")

        hidden_inputs = {
            i.get("name"): i.get("value", "")
            for i in form.find_all("input", type="hidden")
            if i.get("name")
        }

        tasks = []
        for header in soup.find_all("h4", class_="block-title"):
            raw_comp = header.get_text(strip=True).replace("Componente de", "")
            comp_name = limpiar_nombre_sistema(raw_comp)

            table = header.find_next("table")
            if not table:
                continue

            headers = [th.get_text(strip=True).lower() for th in table.find("thead").find_all("th")]
            col_indices = {}
            for i, h in enumerate(headers):
                if "spss" in h:
                    col_indices[i] = "SPSS"
                elif "catal" in h:
                    col_indices[i] = "Catalogo"

            hierarchy = []
            ffq_root = None  # <-- para fijar el root del bloque FFQ

            def _norm(s: str) -> str:
                s = "".join(
                    c for c in unicodedata.normalize("NFD", str(s))
                    if unicodedata.category(c) != "Mn"
                )
                return s.lower().strip()

            for row in table.find("tbody").find_all("tr"):
                cells = row.find_all("td")
                if not cells:
                    continue

                level = self._get_indent_level(cells[0])
                dataset_name = limpiar_nombre_sistema(cells[0].get_text())

                # =========================
                # PATCH: Normalización FFQ
                # =========================
                dn = _norm(dataset_name)

                # 1) Detecta el root del bloque FFQ y lo fija como nivel 0
                if "cuestionarios de frecuencia de consumo de alimentos" in dn:
                    ffq_root = dataset_name
                    hierarchy = []
                    level = 0

                # 2) Fuerza que los grupos por edad estén al mismo nivel (nivel 1) bajo ese root
                elif dn.startswith("frecuencia de consumo de alimentos de "):
                    if ffq_root:
                        hierarchy = [ffq_root]
                        level = 1
                # =========================

                hierarchy = hierarchy[:level] + [dataset_name]

                for col_idx, tipo_archivo in col_indices.items():
                    if col_idx < len(cells):
                        btn = cells[col_idx].find("button")
                        if btn and btn.get("name"):
                            path_parts = [comp_name] + hierarchy + [tipo_archivo]
                            hint = btn.get("title", "").replace("Descargar archivo:", "").strip()
                            if not hint:
                                ext = ".sav" if tipo_archivo == "SPSS" else ".xlsx"
                                hint = f"{dataset_name}{ext}"

                            tasks.append({
                                "path": os.path.join(*path_parts),
                                "post_id": btn["name"],
                                "file_name": limpiar_nombre_sistema(hint),
                                "hidden": hidden_inputs,
                                "type": tipo_archivo
                            })
        return tasks

    def download_file(self, task):
        full_dir = os.path.join(self.output_dir, task["path"])
        os.makedirs(full_dir, exist_ok=True)
        file_path = os.path.join(full_dir, task["file_name"])

        if os.path.exists(file_path) and os.path.getsize(file_path) > 0:
            return "Existe", file_path

        payload = task["hidden"].copy()
        payload[task["post_id"]] = "Accionar"

        try:
            with self.session.post(self.base_url, data=payload, stream=True) as r:
                r.raise_for_status()
                with open(file_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)

            if zipfile.is_zipfile(file_path):
                try:
                    with zipfile.ZipFile(file_path) as z:
                        z.extractall(full_dir)
                except:
                    pass
            return "Bajado", file_path
        except Exception as e:
            return "Error", str(e)

# ==============================================================================
# 4. GENERADOR DE DICCIONARIO + VALIDACIÓN DE MATRIZ (MEJORADO)
# ==============================================================================
def exportar_diccionario_completo(base_dir, out_xlsx):
    rows_vars = []

    archivos_sav = []
    for r, d, files in os.walk(base_dir):
        for file in files:
            if file.lower().endswith(".sav") and not file.startswith("._"):
                archivos_sav.append(os.path.join(r, file))

    if not archivos_sav:
        print("❌ No hay archivos .sav para analizar.")
        return

    print(f"\n📚 Generando diccionario con VALIDACIÓN DE DATOS ({len(archivos_sav)} archivos)...")

    for ruta_sav in tqdm(archivos_sav, desc="Auditando matrices"):
        try:
            # Leemos datos y metadatos
            df, meta = pyreadstat.read_sav(ruta_sav)

            # MODULO (NUTRICIÓN/SALUD) según ruta
            modulo = detectar_modulo(ruta_sav, base_dir)

            nombre_archivo = os.path.basename(ruta_sav)
            padre = os.path.dirname(ruta_sav)

            if os.path.basename(padre).upper() in ['SPSS', 'CATALOGO', 'CATÁLOGO']:
                raw_comp = os.path.basename(os.path.dirname(padre))
            else:
                raw_comp = os.path.basename(padre)

            nombre_encuesta = restaurar_ortografia(raw_comp)

            for col_name in meta.column_names:
                # 1) Catálogo (Diseño)
                pregunta = meta.column_names_to_labels.get(col_name, "")
                etiquetas_dict = meta.variable_value_labels.get(col_name, {})

                texto_catalogo = ""
                if etiquetas_dict:
                    items = [f"{k}={v}" for k, v in etiquetas_dict.items()]
                    texto_catalogo = " | ".join(items)
                    if len(texto_catalogo) > 3000:
                        texto_catalogo = texto_catalogo[:3000] + "... (TRUNCADO)"

                # 2) Valores reales (Matriz) + validación
                status_validacion = "OK"
                try:
                    unicos_reales = sorted(df[col_name].dropna().unique().tolist())
                    valores_encontrados = ", ".join(map(str, unicos_reales))

                    if len(valores_encontrados) > 1000:
                        valores_encontrados = valores_encontrados[:1000] + "... (TRUNCADO)"

                    # Validación: si hay catálogo, verificar que los reales estén en el catálogo
                    if etiquetas_dict:
                        codigos_permitidos = list(etiquetas_dict.keys())
                        errores = []
                        for valor in unicos_reales:
                            if valor not in codigos_permitidos:
                                errores.append(str(valor))
                        if errores:
                            status_validacion = f"⚠️ EXTRAÑOS: {', '.join(errores)}"

                except Exception:
                    valores_encontrados = "Error leyendo datos"
                    status_validacion = "Error Lógica"

                rows_vars.append({
                    "MODULO": modulo,
                    "COMPONENTE": nombre_encuesta,
                    "ARCHIVO": nombre_archivo,
                    "VARIABLE": col_name,
                    "ETIQUETA_PREGUNTA": pregunta,
                    "CATÁLOGO_OFICIAL": texto_catalogo,        # Lo que debería haber
                    "VALORES_EN_MATRIZ": valores_encontrados,  # Lo que realmente hay
                    "VALIDACIÓN": status_validacion,           # ¿Coinciden?
                    "TIPO_DATO": meta.readstat_variable_types.get(col_name, "")
                })

            del df
            del meta

        except Exception as e:
            print(f"⚠️ Error fatal en archivo {os.path.basename(ruta_sav)}: {e}")

    if rows_vars:
        df_export = pd.DataFrame(rows_vars)
        print(f"\n💾 Guardando reporte auditado en: {out_xlsx}")

        # Reordenar columnas para mejor lectura
        cols = ["MODULO", "COMPONENTE", "ARCHIVO", "VARIABLE", "VALIDACIÓN",
                "VALORES_EN_MATRIZ", "CATÁLOGO_OFICIAL", "ETIQUETA_PREGUNTA", "TIPO_DATO"]
        df_export = df_export[cols]

        df_export.to_excel(out_xlsx, index=False)
        print("✅ ¡Proceso terminado!")
        return df_export
    else:
        print("⚠️ No se generó información.")

# ==============================================================================
# 5. EJECUCIÓN
# ==============================================================================
if __name__ == "__main__":
    # 1) DESCARGA
    print("--- 1. INICIANDO DESCARGA ---")
    downloader = EnsanutDownloader(BASE_URL, CARPETA_SALIDA)
    try:
        tasks = downloader.get_tasks()
        print(f"📋 Archivos a procesar: {len(tasks)}")

        with tqdm(total=len(tasks), desc="Descargando") as pbar:
            for task in tasks:
                status, path = downloader.download_file(task)
                pbar.update(1)
    except Exception as e:
        print(f"❌ Error en descarga: {e}")

    # 2) AUDITORÍA Y DICCIONARIO
    print("\n--- 2. CREANDO DICCIONARIO AUDITADO ---")
    ruta_excel = f"{CARPETA_SALIDA}/Diccionario_Validado_ENSANUT_{ANIO_ENCUESTA}.xlsx"
    df_export = exportar_diccionario_completo(CARPETA_SALIDA, ruta_excel)


--- 1. INICIANDO DESCARGA ---
📡 Conectando a https://ensanut.insp.mx/encuestas/ensanutcontinua2023/descargas.php...
📋 Archivos a procesar: 36


Descargando: 100%|██████████| 36/36 [00:16<00:00,  2.18it/s]



--- 2. CREANDO DICCIONARIO AUDITADO ---

📚 Generando diccionario con VALIDACIÓN DE DATOS (36 archivos)...


Auditando matrices: 100%|██████████| 36/36 [00:16<00:00,  2.12it/s]



💾 Guardando reporte auditado en: ENSANUT_2023_Completa/Diccionario_Validado_ENSANUT_2023.xlsx
✅ ¡Proceso terminado!


In [3]:
df_export

,MODULO,COMPONENTE,ARCHIVO,VARIABLE,VALIDACIÓN,VALORES_EN_MATRIZ,CATÁLOGO_OFICIAL,ETIQUETA_PREGUNTA,TIPO_DATO
0,SALUD,Cuestionario de salud de niños (0 a 9 años),menores_ensanut2023_w_n.sav,FOLIO_INT,OK,"2023_01001003_06, 2023_01001003_08, 2023_01001...",,Folio de integrante,string
1,SALUD,Cuestionario de salud de niños (0 a 9 años),menores_ensanut2023_w_n.sav,FOLIO_I,OK,"2023_01001003, 2023_01001006, 2023_01001016, 2...",,Folio,string
2,SALUD,Cuestionario de salud de niños (0 a 9 años),menores_ensanut2023_w_n.sav,hora_ini_1,OK,"00:06:49, 00:35:45, 00:41:46, 00:41:55, 00:42:...",,Hora de Inicio 1,string
3,SALUD,Cuestionario de salud de niños (0 a 9 años),menores_ensanut2023_w_n.sav,fecha_ini_1,OK,"01/09/2023, 01/10/2023, 01/11/2023, 02/09/2023...",,Fecha de Inicio 1,string
4,SALUD,Cuestionario de salud de niños (0 a 9 años),menores_ensanut2023_w_n.sav,hora_fin_1,OK,"00:06:52, 00:09:59, 00:35:43, 00:51:48, 00:52:...",,Hora termino 1,string
...,...,...,...,...,...,...,...,...,...
3958,NUTRICIÓN,Actividad física - Niños (10 a 14 años),actividad_fisica_w_niños.sav,ponde_f,OK,"674.3327692969166, 776.2230885396132, 836.0296...",,Ponderador,double
3959,NUTRICIÓN,Actividad física - Niños (10 a 14 años),actividad_fisica_w_niños.sav,estrato,OK,"1.0, 2.0, 3.0",1.0=Rural ( <2500 Hab ) | 2.0=Urbano ( 2500-9...,Estrato urbanidad/ruralidad,double
3960,NUTRICIÓN,Actividad física - Niños (10 a 14 años),actividad_fisica_w_niños.sav,est_sel,OK,"81.0, 82.0, 83.0, 93.0, 111.0, 112.0, 113.0, 1...",,Estrato de seleccion,double
3961,NUTRICIÓN,Actividad física - Niños (10 a 14 años),actividad_fisica_w_niños.sav,upm,OK,"0100100013312, 0200400013107, 0300300011998, 0...",,Unidad Primaria de Muestreo,string


# **REPORTE DE COBERTURA DE VARIABLES CON SECCIONES**

Ensanut tiene algunos cuestionarios largos con secciones se está intentando agrupar las varibales en esas mismas secciones.

In [4]:
import pandas as pd

# -------------------------------------------------------------------------
# 1. CARGA DE DICCIONARIOS (Copiados exactamente de tu input)
# -------------------------------------------------------------------------

salud_de_ninos = {
  'I. INDICADORES POSITIVOS/RIESGO': ['m0101_id','m0102','m0103_id','m0104a','m0104b','M0104_1','M0104_2','m0105','m0106','m0107a','m0107aa','m0107ab','m0107ac','m0107b','m0107c','m0107d','m0107e','m0107f','m0107g','m0107h','m0107i','m0108','m0109','m0110','m0111a','m0111aa','m0111ae','m0111b','m0111be','m0111d','m0111e','m0111m','m0112a','m0112aa','m0112b','m0112b1','m0112b2','m0112b3','m0112b4','m0112b5','m0112c','m0112d','m0112e','m0112f','m0112g','m0112h','m0112i','m0112j','m0112k','m0112l','m0112m','m0113','m0114a','m0114b','m0114d','m0115','m0116','m0117','M0118A','M0118B','M0118C','M0118D','M0118E','m0118esp'],
  'II. EDUCACIÓN': ['m0201','m0202','M0203A','M0203B','M0203C','M0203D','M0203E','m0203esp','M0203F','M0203G','m0204','m0205','M0206A','M0206B','M0206C','M0206D','M0206E','m0206esp','M0206F','M0206G'],
  'III. DESARROLLO INFANTIL': ['m0301','m0302','m0303','m0304','m0305','m0306','m0307','m0308','m0309','m0310','m0311','m0312','m0313','m0314','m0315','m0316','m0317','m0318','m0319'],
  'IV. CALIDAD DEL CONTEXTO': ['m0401','m0402a','m0402a_esp','m0402b','m0402b_esp','m0402c','m0402c_esp','m0403a','m0403b','M0404AA','M0404AB','M0404AC','M0404BA','M0404BB','M0404BC','M0404CA','M0404CB','M0404CC','M0404DA','M0404DB','M0404DC','M0404EA','M0404EB','M0404EC','M0404FA','M0404FB','M0404FC','M0404GA','M0404GB','M0404GC','m0405a','m0405b','m0405c','m0405d','m0405e','m0405f','m0405g','m0405h','m0405i','m0405j','m0405k','m0406'],
  'V. VACUNACIÓN NIÑOS': ['m0501','m0501esp','m0502','m0503','m0504','m0505','m0505esp','m0506','m0507','m0508','m0509','m0510','m0511','m0512','m0512v','m0513','m0513v','m0514','m05141','m0514v','m0514v1','m0515','m0515v','m0516','m0516v','m0517','m0517v','m0518','m0518v','m0520','m0520v','m0521','m0521v','m0522a','m0522av','m0522a_esp','m0522b','m0522bv','m0522b_esp','m0522c','m0522cv','m0522c_esp','m0524','m0524d','m0525d_1','m0525d_2','m0525d_3','m0525d_4','m0525d_5','m0525_1','m0525_2','m0525_3','m0525_4','m0525_5','m05261d_1','m05261d_2','m05261d_3','m05261d_4','m05261_1','m05261_2','m05261_3','m05261_4','m0526d_1','m0526d_2','m0526d_3','m0526d_4','m0526_1','m0526_2','m0526_3','m0526_4','m0527','m0527d','m05281d_2','m05281_1','m05281_2','m0528d1_1','m0528d_1','m0528d_2','m0528d_3','m0528_1','m0528_2','m0528_3','m0529d_1','m0529d_2','m0529d_3','m0529_1','m0529_2','m0529_3','m0530d_1','m0530d_2','m0530d_3','m0530d_4','m0530d_5','m0530d_6','m0530_1','m0530_2','m0530_3','m0530_4','m0530_5','m0530_6','m0531d_1','m0531d_2','m0531_1','m0531_2','m0534do_1','m0534do_2','m0534do_3','m0534do_4','m0534do_5','m0534d_1','m0534d_2','m0534d_3','m0534d_4','m0534d_5','m0534n_1','m0534n_2','m0534n_3','m0534n_4','m0534n_5','m0534_1','m0534_2','m0534_3','m0534_4','m0534_5','m0535','m0536','m0537','m0538','m0539'],
  'VI. ENF. DIARREICAS': ['m0601','m0602','m0603','m0604','m0605','M0606A','m0606aesp','M0606B','M0606C','m0607a','m0607aesp','m0607b','m0607besp','m0607c','m0607cesp','m0608','M0609A','M0609B','M0609C','M0609D','M0609E','m0609esp','m0610a','m0611','M0613A','M0613B','M0613C','M0613D','M0613E','m0613esp','M0614A','M0614B','M0614C','M0614D','M0614E1','m0614esp'],
  'VII. INFECCIONES RESPIRATORIAS': ['m0701','m0702','m0703','m0704','m0704esp','m0705','M0706A','m0706aesp','M0706B','M0706C','m0707a','m0707aesp','m0707b','m0707besp','m0707c','m0707cesp','m0709a','M0712A','M0712B','M0712C','M0712D','M0712E','m0712esp','m0713','m0714','m0715'],
  'VIII. ACCIDENTES': ['m0801','m0802','m0803','m0804','m0805','m0805esp','m0806','m0806esp','m0807','m0807esp','m0808','m0808esp'],
  'IX. FUNCIONAMIENTO': ['m0901','m0902','m0903','m0904','m0905','m0906','m0907','m0908','m0909','m0910','m0911','m0912','m0913','m0914','m0915','m0916','m0917','m0918','m0919','m0920','m0921','m0922','m0923','m0924','m0925','m0926','m0927','m0928','m0929','m0930','m0931','m0932','m0933','m0934','m0935','m0936','m0937','m0938'],
}

salud_de_adolescentes = {
    'I. FACTORES DE RIESGO': ['d0101', 'd0102', 'd0103', 'd0104', 'd0105', 'd0106p', 'd0106t', 'd0107', 'd0108', 'd0109', 'd0110', 'd0111', 'd0112'],
    'I.A USO DE SUSTANCIAS': ['d01a01', 'D01A02A', 'D01A02B', 'D01A02C', 'D01A02D', 'd01a02e', 'D01A02E1', 'D01A02F', 'D01A02G', 'D01A02H', 'D01A02I', 'D01A02J', 'd01a03a', 'd01a03b', 'd01a03c', 'd01a03d', 'd01a04a', 'd01a04b', 'd01a04c', 'd01a04d', 'd01a05a', 'd01a05b', 'd01a05c', 'd01a05d', 'd01a06a', 'd01a06b', 'd01a06c', 'd01a06d', 'd01a07a', 'd01a07b', 'd01a07c', 'd01a07d', 'd01a08a', 'd01a08b', 'd01a08c', 'd01a08d', 'd01a08e', 'd01a08f', 'd01a08g', 'd01a08h', 'd01a08i', 'd01a09a', 'd01a09b', 'd01a09c', 'd01a09d', 'd01a09e', 'd01a09f', 'd01a09g', 'd01a09h', 'd01a10a', 'd01a10b', 'd01a10c', 'd01a10d', 'd01a10e', 'd01a10f', 'd01a10g', 'd01a10h', 'd01a11af', 'd01a11av', 'd01a11bf', 'd01a11bv', 'd01a11cf', 'd01a11cv', 'd01a11df', 'd01a11dv', 'd01a11ef', 'd01a11ev', 'd01a11ff', 'd01a11fv', 'd01a11gf', 'd01a11gv', 'd01a11hf', 'd01a11hv'],
    'II. ENFERMEDADES CRÓNICAS': ['d02o1a', 'd02o1b', 'd02o1c', 'd02o1d'],
    'III. SALUD SEXUAL Y REPRODUCTIVA': ['d0301', 'D0302A', 'D0302B', 'D0302C', 'D0302D', 'D0302E', 'D0302F', 'D0302G', 'D0302H', 'D0302I', 'D0302J', 'D0302K', 'D0302L', 'D0302M', 'D0302N', 'd0303', 'd0304', 'd0304e', 'd0305', 'd0306', 'd0306a', 'd0306b', 'D0306CA', 'D0306CB', 'D0306CC', 'D0306CD', 'd0306ce', 'D0306CE1', 'D0306CF', 'D0306CG', 'D0306CH', 'd0306d', 'd0306de', 'd0306e', 'd0306f', 'd0306fe', 'D0307A', 'D0307B', 'D0307C', 'D0307D', 'D0307E', 'D0307F', 'D0307G', 'D0307H', 'D0307I', 'D0307J', 'D0307K', 'D0307L', 'd0308', 'd0308e', 'd0309', 'd0309a', 'D0311A', 'D0311B', 'D0311C', 'D0311D', 'D0311E', 'D0311F', 'D0311G', 'D0311H', 'D0311I', 'D0311J', 'D0311K', 'D0311L', 'D0311M', 'D0311N', 'd0312', 'd0313', 'd0314', 'd0315', 'd0316a', 'd0316b', 'd0316c', 'd0316d', 'd0316e', 'd0317a', 'd0317d', 'd0317m', 'd0318', 'd0319', 'd0319e', 'd0320', 'd0321a', 'd0321b', 'd0321c', 'd0321d', 'd0321e', 'd0321f', 'd0321g', 'd0321h', 'd0321i', 'd0321j', 'd0321k', 'd0321l', 'd0321m', 'd0321n', 'd0321nn', 'd0321o', 'd0321p', 'd0322', 'd0322e', 'd0323', 'd0324', 'd0325', 'd0325a', 'd0327a', 'd0327b', 'd0327c', 'd0327d', 'd0327e', 'd0327f', 'd0327g', 'd0327h', 'd0327i', 'd0327j', 'd0328', 'd0328a', 'd0329', 'd0330', 'd0331'],
    'IV. FUNCIONAMIENTO': ['d0401', 'd0402', 'd0403', 'd0404a', 'd0404b', 'd0405a', 'd0405b', 'd0406', 'd0407', 'd0408', 'd0409', 'd0410', 'd0411', 'd0412', 'd0413', 'd0414', 'd0415', 'd0416', 'd0417', 'd0418', 'd0419', 'd0420', 'd0421', 'd0422', 'd0423a', 'd0423b', 'd0424a', 'd0424b', 'd0425', 'd0426', 'd0427', 'd0428'],
    'V. VACUNACIÓN': ['d0501', 'd05021', 'd05022', 'd05031', 'd05032', 'd05041', 'd05042', 'd05051', 'd05052', 'd05061', 'd05062', 'd05071', 'd05072', 'd0508', 'd0509', 'd0510pa', 'd0510pb', 'd0510sa', 'd0510sb', 'd0511pa', 'd0511pb', 'd0511ra', 'd0511rb', 'd0511sa', 'd0511sb', 'd0511ta', 'd0511tb', 'd0512ua', 'd0512ub', 'd0513ua', 'd0513ub', 'd0514pa', 'd0514pb', 'd0514sa', 'd0514sb', 'd0514ua', 'd0514ub', 'd0515pa', 'd0515pb', 'd0515sa', 'd0515sb', 'd0515ta', 'd0515tb', 'd0516oa', 'd0516oa1', 'd0516oa2', 'd0516oa3', 'd0516ob', 'd0516ob1', 'd0516ob2', 'd0516ob3', 'd0516oc', 'd0516oc1', 'd0516oc2', 'd0516oc3', 'd0516od', 'd0516od1', 'd0516od2', 'd0516od3', 'd0516oe', 'd0516oe1', 'd0516oe2', 'd0516oe3'],
    'VI. SINTOMATOLOGÍA DEPRESIVA': ['d0601a', 'd0601b', 'd0601c', 'd0601d', 'd0601e', 'd0601f', 'd0601g'],
    'VI.A CONDUCTAS ALIMENTARIAS': ['d06a1', 'd06a10', 'd06a2', 'd06a3', 'd06a4', 'd06a5', 'd06a6', 'd06a7', 'd06a8', 'd06a9'],
    'VII. ACCIDENTES': ['d0701', 'd0702', 'd0703', 'd0704', 'd0705', 'd0707', 'd0707e', 'd0708', 'd0708e', 'd0709'],
    'VIII. ATAQUE O VIOLENCIA': ['d0801', 'D0802A', 'D0802B', 'D0802C', 'D0802D', 'd0802e', 'D0802E1', 'D0802F', 'D0802G', 'D0802H', 'D0802I', 'D0802J', 'D0802K', 'D0802L', 'd0803', 'd0803e', 'd0804', 'd0804e', 'd0806', 'd0807', 'd0807e', 'd0808', 'd0809', 'd0810', 'd0811', 'd0812', 'd0813', 'd0813e', 'd0814', 'd0815', 'd0815e', 'd0816', 'd0816e', 'd0817', 'd0818', 'd0819', 'd0820', 'D0821A', 'D0821B', 'D0821C', 'D0821D', 'd0821e', 'D0821E1', 'D0821F', 'D0821G', 'D0821H', 'D0821I', 'D0821J', 'D0821K', 'D0821L', 'd0822'],
    'IX. DISCIPLINA': ['d0901a', 'd0901b', 'd0901c', 'd0901d', 'd0901e', 'd0901f', 'd0901g', 'd0901h', 'd0901i', 'd0901j', 'd0901k', 'd0902']
}

salud_adultos_20_mas = {
  'I. SOBREPESO Y OBESIDAD': ['a01esp', 'a0104', 'a0107', 'a0108', 'a0109', 'A0110A', 'A0110B', 'A0110C', 'A0110D', 'A0110E', 'A0110F', 'A0110G', 'A0110H', 'A0110I'],
  'II. SINTOMATOLOGÍA DEPRESIVA': ['a0202', 'a0203', 'a0203e', 'a0204', 'a0205', 'a0206', 'a0206e', 'a0207', 'a0211', 'a0212', 'a0213', 'a0214', 'a0215', 'a0216', 'a0217'],
  'III. DIABETES MELLITUS': ['a0301', 'a0301a', 'a0302', 'a0303num', 'a0304a', 'a0304b', 'a0304be', 'a0304d', 'a0304m', 'a0305', 'a0305esp', 'a0306a', 'a0306b', 'a0306c', 'a0306d', 'a0306e', 'a0306f', 'a0306g', 'a0306h', 'a0306i', 'a0306j', 'a0306k', 'a0306l', 'a0306m', 'a0306n', 'a0306o', 'a0307', 'a0308a', 'a0308m', 'a0309a', 'a0309m', 'a0310', 'a0310a', 'A0312A', 'A0312B', 'A0312C', 'A0312D', 'a0313', 'a0313a', 'a0314', 'A0314AA', 'A0314AB', 'A0314AC', 'A0314AD', 'A0314AE', 'A0314AF', 'A0314AG', 'A0314AH', 'A0315A', 'A0315B', 'A0315C', 'A0315D', 'A0315E', 'A0315F', 'A0315G', 'A0315H', 'A0315I', 'A0315J', 'A0315K', 'A0315L', 'A0315M', 'A0315N', 'A0315O', 'A0315P', 'A0315Q', 'A0315R', 'A0315S', 'A0315T', 'A0315U', 'A0315V', 'a0316a', 'a0316b', 'a0316d', 'a0316e', 'a0316f', 'a0316g', 'a0316h', 'a0316i', 'a0316j', 'a0316k', 'a0316l', 'a03061a', 'a03061aa', 'a03061b', 'a03061bb', 'a03061c', 'a03061cc', 'a03061j', 'a03061o'],
  'IV. HIPERTENSIÓN ARTERIAL': ['a0401', 'a0402a', 'a0404', 'a0404a', 'a0405a', 'a0405aa', 'a0405b', 'a0405c', 'A0405DA', 'A0405DB', 'A0405DC', 'A0405DD', 'A0405DE', 'A0405DF', 'A0405DG', 'A0405DH', 'a0405m', 'a0406', 'a0406b', 'a0406e', 'a0407', 'a0407esp', 'A0408A', 'A0408B', 'A0408C', 'A0408D', 'A0408E', 'a0409', 'a0409a', 'a0410a', 'a0410b', 'a0410c', 'a0410d', 'a0410e', 'a0410ev', 'a0410f', 'a0410fd', 'a0410fv'],
  'V. ENFERMEDAD CARDIOVASCULAR': ['a0502a', 'a0502b', 'a0502c', 'a0502d'],
  'VI. ENF. RENAL/HIPERCOLESTEROLEMIA': ['a0601a', 'a0601b', 'a0601c', 'a0603', 'a0604', 'A0605A', 'A0605B', 'A0605C', 'A0605D', 'a0606', 'a0607a', 'A0607A1', 'A0607B', 'A0607C', 'A0607D'],
  'VII. ANTECEDENTES FAMILIARES': ['a07her', 'a0701h', 'a0701m', 'a0701p', 'a0702h', 'a0702m', 'a0702p', 'a0703h', 'a0703m', 'a0703p', 'a0704h', 'a0704m', 'a0704p'],
  'VIII. SALUD SEXUAL Y REPRODUCTIVA': ['a0801', 'a0802', 'A0803A', 'A0803B', 'A0803C', 'A0803D', 'A0803E', 'A0803F', 'A0803G', 'A0803H', 'A0803I', 'A0803J', 'A0803K', 'A0803L', 'A0803M', 'A0803N', 'A0803O', 'A0804A', 'A0804B', 'A0804C', 'A0804D', 'A0804E', 'A0804F', 'A0804G', 'A0804H', 'A0804I', 'A0804J', 'A0804K', 'A0804L', 'A0804M', 'A0804N', 'A0804O', 'A0804P', 'A0804Q', 'a0805', 'a0806', 'a0807', 'a0808', 'a0809', 'a0810a', 'a0810ad', 'a0810ae', 'a0810b', 'a0810c', 'a0811a', 'a0811d', 'a0811m', 'a0812', 'a0813', 'a0813esp', 'a0814', 'a0815a', 'a0815b', 'a0815c', 'a0815d', 'a0815e', 'a0815f', 'a0815g', 'a0815h', 'a0815i', 'a0815j', 'a0815k', 'a0815l', 'a0815m', 'a0815n', 'a0815o', 'a0815p', 'a0815q', 'a0816', 'a0816esp', 'a0817', 'a0818', 'a0819f', 'a0819k', 'a0821a', 'a0821b', 'a0821c', 'a0821d', 'a0821e', 'a0821f', 'a0821g', 'a0821h', 'a0821i', 'a0821j', 'a0822', 'a0823', 'a0824', 'a0825'],
  'IX. VACUNACION ADULTOS': ['a0901', 'a0902', 'a0903', 'a0904', 'a0906', 'a0908', 'a0914', 'a0915', 'a0916', 'a0917', 'a0918', 'a0919', 'a0920', 'a0921', 'a0995', 'a0997', 'a09091a', 'a09091bd', 'a09092a', 'a09092bd', 'a09093a', 'a09093bd', 'a09101a', 'a09101bd', 'a09102a', 'a09102bd', 'a09103a', 'a09103bd', 'a09104a', 'a09104bd', 'a09111a', 'a09111bd', 'a09121a', 'a09121bd', 'a09122a', 'a09122bd', 'a09131bd', 'a09131ed', 'a09131nom', 'a09132bd', 'a09132edad', 'a09132nom', 'a09133bd', 'a09133ed', 'a09133nom', 'a09134bd', 'a09134ed', 'a09134nom', 'a09135bd', 'a09135ed', 'a09135nom', 'a09221a', 'a09221bd', 'a09222a', 'a09222bd', 'a09223a', 'a09223bd', 'a09231a', 'a09231bd', 'a09232a', 'a09232bd', 'a09233a', 'a09233bd', 'a09234a', 'a09234bd', 'a09241a', 'a09241bd', 'a09251bd', 'a09251ed', 'a09251nom', 'a09252bd', 'a09252ed', 'a09252nom', 'a09253bd', 'a09253ed', 'a09253nom', 'a09254bd', 'a09254ed', 'a09254nom', 'a09255bd', 'a09255ed', 'a09255nom'],
  'X. PROGRAMAS PREVENTIVOS': ['a1001a', 'a1001ab', 'a1001b', 'a1001bb', 'a1001c', 'a1001cb', 'a1001d', 'a1001db', 'a1001f', 'a1001f1', 'a1001f1b', 'a1001fb', 'a1001g', 'a1001gb', 'a1001h', 'a1001hb', 'a1001i', 'a1001ib', 'a1001j', 'a1001jb', 'a1002a', 'a1002ae', 'a1002b', 'a1002be', 'a1002c', 'a1002ce', 'a1002d', 'a1002de', 'a1002f', 'a1002f1', 'a1002f1e', 'a1002fe', 'a1002g', 'a1002ge', 'a1002h', 'a1002he', 'a1002i', 'a1002ie', 'a1002j', 'a1002je', 'a1003a', 'a1003b', 'a1003c', 'a1003d', 'a1003f', 'a1003f1', 'a1003g', 'a1003h', 'a1003i', 'a1003j', 'a1004a', 'a1004b', 'a1004c', 'a1004d', 'a1004f', 'a1004f1', 'a1004g', 'a1004h', 'a1004i', 'a1004j', 'a1005a', 'a1005b', 'a1005c', 'a1005d', 'a1005f', 'a1005f1', 'a1005g', 'a1005h', 'a1005i', 'a1005j', 'a1006a', 'a1006ab', 'a1006abe', 'a1006b', 'a1006bb', 'a1006bbe', 'a1006c', 'a1006cb', 'a1006cbe', 'a1006d', 'a1006db', 'a1006dbe', 'a1006f', 'a1006f1', 'a1006f1b', 'a1006f1be', 'a1006fb', 'a1006fbe', 'a1006g', 'a1006gb', 'a1006gbe', 'a1006h', 'a1006hb', 'a1006hbe', 'a1006i', 'a1006ib', 'a1006ibe', 'a1006j', 'a1006jb', 'a1006jbe', 'a1007a', 'a1007b', 'a1007c', 'a1007d', 'a1007f', 'a1007f1', 'a1007g', 'a1007h', 'a1007i', 'a1007j', 'a1008a', 'a1008b', 'a1008c', 'a1008d', 'a1008f', 'a1008f1', 'a1008g', 'a1008h', 'a1008i', 'a1008j'],
  'XI. ACCIDENTES': ['a1101', 'a1102', 'a1103', 'a1104', 'a1105', 'a1107', 'a1107e', 'a1108', 'a1108e', 'a1109', 'a1109e'],
  'XII. ATAQUE Y VIOLENCIA': ['a1201', 'A1202A', 'A1202B', 'A1202C', 'A1202D', 'a1202e', 'A1202E1', 'A1202F', 'A1202G', 'A1202H', 'A1202I', 'A1202J', 'A1202K', 'a1203', 'a1203e', 'a1204', 'a1204e', 'a1206', 'a1206e', 'a1207', 'a1207e', 'a1208', 'a1208e', 'a1209', 'a1210', 'a1211', 'a1212', 'a1213', 'a1214'],
  'XIII. FACTORES DE RIESGO': ['a13a1', 'a13a10a', 'a13a10b', 'a13a10c', 'a13a10d', 'a13a10e', 'a13a10f', 'a13a10g', 'a13a10h', 'a13a11af', 'a13a11av', 'a13a11bf', 'a13a11bv', 'a13a11cf', 'a13a11cv', 'a13a11df', 'a13a11dv', 'a13a11ef', 'a13a11ev', 'a13a11ff', 'a13a11fv', 'a13a11gf', 'a13a11gv', 'a13a11hf', 'a13a11hv', 'A13A2A', 'A13A2B', 'A13A2C', 'A13A2D', 'a13a2e', 'A13A2E1', 'A13A2F', 'A13A2G', 'A13A2H', 'A13A2I', 'A13A2J', 'a13a3a', 'a13a3b', 'a13a3c', 'a13a3d', 'a13a4a', 'a13a4b', 'a13a4c', 'a13a4d', 'a13a5a', 'a13a5b', 'a13a5c', 'a13a5d', 'a13a6a', 'a13a6b', 'a13a6c', 'a13a6d', 'a13a7a', 'a13a7b', 'a13a7c', 'a13a7d', 'a13a8a', 'a13a8b', 'a13a8c', 'a13a8d', 'a13a8e', 'a13a8f', 'a13a8g', 'a13a8h', 'a13a8i', 'a13a9a', 'a13a9b', 'a13a9c', 'a13a9d', 'a13a9e', 'a13a9f', 'a13a9g', 'a13a9h', 'a1301', 'a1302', 'a1303', 'a1304', 'a1305', 'a1306p', 'a1306t', 'a1307', 'a1308', 'a1309', 'a1310', 'a1311', 'a1312', 'a1313', 'a1314'],
  'XIV. FUNCIONAMIENTO': ['a1401', 'a1402', 'a1403a', 'a1403b', 'a1404a', 'a1404b', 'a1405', 'a1406', 'a1407', 'a1408'],
  'XV. EVALUACIÓN DE MEMORIA': ['a1501', 'a1502', 'a1503', 'a1504', 'a1505', 'a1506', 'a1507', 'a1508'],
  'XVI. ACTIVIDADES VIDA DIARIA': ['a1601', 'a1602', 'a1603', 'a1604', 'a1605', 'a1606'],
  'XVII. ACTIVIDADES INSTRUMENTALES': ['a1701', 'a1702', 'a1703', 'a1704', 'a1705', 'a1706', 'a1707', 'a1708'],
  'XVIII. CAÍDAS Y FRACTURAS': ['a1801', 'a1801a', 'a1802', 'a1803', 'a1803e', 'a1804', 'a1804a', 'a1805', 'a1806', 'a1806e', 'a1807'],
  'XIX. SARCOPENIA (SARC-F)': ['a1901', 'a1902', 'a1903', 'a1904'],
}

Utilizadores_servicios = {
  'I. UTILIZACIÓN': ['u0101','u0103'],
  'II. ATENCIÓN': ['u0201','u0201e','u0202a','u0202b','u0202b1','u0202b1e','u0202be','u0202c1a','u0202c1b','u0202c1c','u0202c1e','u0202ca','u0202cb','u0202cc','u0202ce','u0202d1a','u0202d1b','u0202d1c','u0202d1e','u0202da','u0202db','u0202dc','u0202de','u0202e','u0202ua','u0202ub','U0202UC','u0203','u0204h','u0204m','u0205h','u0205m','u0206h','u0206m','u0207','u0208'],
  'III. MEDICAMENTOS': ['u0301','u0303','U0304A','U0304B','U0304C','U0304D1','U0304E1','U0304E','u0306'],
  'IV. ESTUDIOS LAB/GABINETE': ['u0401','u0402','U0403A','U0403B','U0403C','U0403D','u0403e','U0403E1','U0403F','u0405','u0406','u0407a','u0407b','u0407c'],
}



indice_bienestar = {"Índice de bienestar":["FOLIO_I","indice1","nse5F","nseF"]}

hogar = {"I. CARACTERÍSTICAS DE LA VIVIENDA":["h0101","h0102","h0103","h0104","h0105","h0106","h0107","h0108","h0109","h0110","h0110esp","h0111","h0112","h0113","h0114","h0118","h0119","h0120","h0121","h0122","h0123","h0123esp","h0124","h0125"],"II. IDENTIFICACIÓN DE HOGARES":["h0201","h0202","h0203","h0204"],"III. CARACTERÍSTICAS SOCIODEMOGRÁFICAS":["h0327"],"V. OTRAS CARACTERÍSTICAS DEL HOGAR":["h0501a","h0501b","h0501c","h0501e","h0501f","h0501g","h0501k","h0501l","h0501n","h0501o","h0501p","h0501q","h0501r","h0501s","h0501t","h0501u","h0501v","h0501w","h0501x","h0501y"],"VII. SEGURIDAD ALIMENTARIA":["h0701","h0702","h0703","h0704","h0705","h0706","h0707","h0708","h0709","h0710","h0711","h0712","h0713","h0714","h0715","h0716"],"VIII. ESCALA DE EXPERIENCIAS DE INSEGURIDAD DEL AGUA EN EL HOGAR":["h0801","h0802","h0803","h0804","h0805","h0806","h0807","h0808","h0809","h0810","h0811","h0812","h08a01","h08a01esp","h08a02","h08a03","h08a0401","h08a0402","h08a0403","h08a0404","h08a0405","h08a0406","h08a0407","h08a05","h08a051","h08a052","h08a06","h08a07","h08a08","h08a091","h08a09a","h08a09b","h08a09c","h08a09d","h08a09e","h08a09f","h08a09g","h08a09h","h08a09i","h08a09ies","h08a101","h08a10a","h08a10b","h08a10c","h08a10d","h08a10e","h08a10es","h08a11","h08a11esp","h08a12","h08a12esp","h08a13","h08a13esp","h08b01","h08b02","h08b02_a","h08b03","h08b03_a","h08b04","h08b05","h08b05_a"],"META/IDENTIFICADORES (no sección)":["FOLIO_I","comentario","completa","cuenta","desc_ent","desc_mun","edad_j","entidad","est_sel","estrato","fecha_fin","fecha_fin_1","fecha_fin_2","fecha_fin_3","fecha_fin_4","fecha_ini_1","fecha_ini_2","fecha_ini_3","fecha_ini_4","hora_fin","hora_fin_1","hora_fin_2","hora_fin_3","hora_fin_4","hora_ini_1","hora_ini_2","hora_ini_3","hora_ini_4","jefe","maquina","municipio","nota1","nota1a","nota2","nota3","nota4","nota5","nota5a","nota6","nota7","nota8","num_int","otroent","ponde_f","resultado_1","resultado_2","resultado_3","resultado_4","sexo_j","tiempo","tiempo1","tiempo2","tiempo3","tiempo4","upm","veri_selec","x_region"],"EXTRA (no aparece en este PDF): H30*":["h305i","h305t"],"EXTRA (no aparece en este PDF): NOTA08*":["nota08a1","nota08a2","nota08a3","nota08a4","nota08b11"],"EXTRA (no aparece en este PDF): NOTA17*":["nota17"]}

residentes = {"III. CARACTERÍSTICAS SOCIODEMOGRÁFICAS":["H0310A","H0310B","H0310C","h0302","h0303","h0304a","h0304d","h0304m","h0305","h0305_es","h0306","h0306e","h0306p","h0307","h0307q","h0308","h0308q","h0309","h0309e","h0310e","h0311","h0312","h0313","h0314","h0316","h0316esp","h0317a","h0317g","h0318","h0319","h0320","h0320q","h0321","h0322","h0323","h0324","h0324esp"],"IV. SITUACIÓN DE SALUD Y UTILIZACIÓN DE SERVICIOS DE SALUD":["H0405A","H0405B","H0405C","H0407A","H0407B","H0407C","H0409A","H0409B","H0409C","H0409D","h0401","h0402","h0402esp","h0403","h0404","h0405esp","h0406","h0407esp","h0408","h0408esp"],"VI. APOYO DE PROGRAMAS SOCIALES":["H0605BAA","H0605BAB","H0605BAC","H0605BAD","H0605BAE1","H0605BAF","H0605BAG","H0605BAH","H0605BAI","H0605BAJ","H0605BAK","H0605BAL","H0605BAM","H0605BB1A","H0605BB1B","H0605BB1C","H0605BB1D","H0605BB1E1","H0605BB1F","H0605BB1G","H0605BB1H","H0605BB1I","H0605BB1J","H0605BB1K","H0605BB1L","H0605BB1M","H0605JAA","H0605JAB","H0605JAC","H0605JAD","H0605JAE1","H0605JAF","H0605JAG","H0605JAH","H0605JAI","H0605JAJ","H0605JAK","H0605JAL","H0605JAM","H0605JB1A","H0605JB1B","H0605JB1C","H0605JB1D","H0605JB1E1","H0605JB1F","H0605JB1G","H0605JB1H","H0605JB1I","H0605JB1J","H0605JB1K","H0605JB1L","H0605JB1M","H0605KAA","H0605KAB","H0605KAC","H0605KAD","H0605KAE1","H0605KAF","H0605KAG","H0605KAH","H0605KAI","H0605KAJ","H0605KAK","H0605KAL","H0605KAM","H0605KB1A","H0605KB1B","H0605KB1C","H0605KB1D","H0605KB1E1","H0605KB1F","H0605KB1G","H0605KB1H","H0605KB1I","H0605KB1J","H0605KB1K","H0605KB1L","H0605KB1M","H0605LAA","H0605LAB","H0605LAC","H0605LAD","H0605LAE1","H0605LAF","H0605LAG","H0605LAH","H0605LAI","H0605LAJ","H0605LAK","H0605LAL","H0605LAM","H0605LB1A","H0605LB1B","H0605LB1C","H0605LB1D","H0605LB1E1","H0605LB1F","H0605LB1G","H0605LB1H","H0605LB1I","H0605LB1J","H0605LB1K","H0605LB1L","H0605LB1M","H0605MAA","H0605MAB","H0605MAC","H0605MAD","H0605MAE1","H0605MAF","H0605MAG","H0605MAH","H0605MAI","H0605MAJ","H0605MAK","H0605MAL","H0605MAM","H0605MB1A","H0605MB1B","H0605MB1C","H0605MB1D","H0605MB1E1","H0605MB1F","H0605MB1G","H0605MB1H","H0605MB1I","H0605MB1J","H0605MB1K","H0605MB1L","H0605MB1M","h0601a","h0601b","h0601c","h0601d","h0601e","h0601f","h0601g","h0601h","h0601i","h0601j","h0601k","h0601l","h0601m","h0601n","h0603a","h0603b","h0603c","h0603d","h0603e","h0603f","h0603g","h0603h","h0603i","h0603j","h0603k","h0603l","h0603m","h0603n","h0604ba","h0604bb","h0605b","h0605ba_esp","h0605bb","h0605bb1_esp","h0605j","h0605ja_esp","h0605jb","h0605jb1_esp","h0605k","h0605ka_esp","h0605kb","h0605kb1_esp","h0605l","h0605la_esp","h0605lb","h0605lb1_esp","h0605m","h0605ma_esp","h0605mb","h0605mb1_esp"],"META/IDENTIFICADORES (no sección)":["FOLIO_I","FOLIO_INT","completa1","desc_ent1","desc_mun1","entidad1","est_sel","estrato","id_int","int_nuevo","intp1","intp3","intp4","meses","municipio1","ponde_f","upm","x_region"],"EXTRA (no aparece en este PDF): H17*":["H1704A","H1704B","H1704C","H1704D","H1704E","H1704F","H1704G","H1704H","H1704I","H1704J","H1704K","H1704L","H1704M","H1704N","H1704O","H1704P","H1704Q","H1704R","H1704S","H1704T","h1701","h1702a","h1702b","h1703","h1705","h1706"]}



# -------------------------------------------------------------------------
# 2. PROCESAMIENTO
# -------------------------------------------------------------------------

# Paso clave: Obtener las variables del DataFrame y convertirlas a minúsculas
# para que la búsqueda sea insensible a mayúsculas/minúsculas.
# Asumimos que df_export ya está cargado como se ve en tu imagen.
df_vars_normalized = set(df_export['VARIABLE'].astype(str).str.lower().str.strip())

def checar_cobertura(nombre_grupo, diccionario):
    reporte = []

    for seccion, variables_esperadas in diccionario.items():
        # Convertir también las variables del diccionario a minúsculas
        esperadas_norm = [v.lower().strip() for v in variables_esperadas]

        # Encontrar intersección (las que sí están)
        encontradas = [v for v in esperadas_norm if v in df_vars_normalized]
        faltantes = [v for v in esperadas_norm if v not in df_vars_normalized]

        total = len(esperadas_norm)
        num_encontradas = len(encontradas)
        porcentaje = (num_encontradas / total * 100) if total > 0 else 0

        reporte.append({
            'Grupo': nombre_grupo,
            'Sección': seccion,
            'Total Variables': total,
            'Encontradas': num_encontradas,
            'Faltantes': len(faltantes),
            'Cobertura (%)': round(porcentaje, 1),
            # 'Lista Faltantes': faltantes # Descomenta si quieres ver cuáles faltan
        })

    return pd.DataFrame(reporte)

# -------------------------------------------------------------------------
# 3. EJECUCIÓN Y REPORTE
# -------------------------------------------------------------------------

df_ninos = checar_cobertura("Niños", salud_de_ninos)
df_adolescentes = checar_cobertura("Adolescentes", salud_de_adolescentes)
df_adultos = checar_cobertura("Adultos", salud_adultos_20_mas)
df_utilizadores = checar_cobertura("Utilizadores", Utilizadores_servicios)




df_indice_bienestar = checar_cobertura("Indice bienestar", indice_bienestar)
df_hogar = checar_cobertura("Info hogar", hogar)
df_residentes = checar_cobertura("Info residentes", residentes)


# Unir todo en un solo reporte final
reporte_final = pd.concat([df_ninos, df_adolescentes, df_adultos, df_utilizadores, df_indice_bienestar,df_hogar, df_residentes], ignore_index=True)

# Mostrar el reporte
print("--- REPORTE DE COBERTURA DE VARIABLES (Case Insensitive) ---")
display(reporte_final)

# Opcional: Si quieres exportar las que faltan a un Excel para revisarlas
# reporte_final.to_excel("reporte_cobertura_variables.xlsx", index=False)

--- REPORTE DE COBERTURA DE VARIABLES (Case Insensitive) ---


,Grupo,Sección,Total Variables,Encontradas,Faltantes,Cobertura (%)
0,Niños,I. INDICADORES POSITIVOS/RIESGO,64,64,0,100.0
1,Niños,II. EDUCACIÓN,20,20,0,100.0
2,Niños,III. DESARROLLO INFANTIL,19,19,0,100.0
3,Niños,IV. CALIDAD DEL CONTEXTO,42,42,0,100.0
4,Niños,V. VACUNACIÓN NIÑOS,129,129,0,100.0
5,Niños,VI. ENF. DIARREICAS,36,36,0,100.0
6,Niños,VII. INFECCIONES RESPIRATORIAS,26,26,0,100.0
7,Niños,VIII. ACCIDENTES,12,12,0,100.0
8,Niños,IX. FUNCIONAMIENTO,38,38,0,100.0
9,Adolescentes,I. FACTORES DE RIESGO,13,13,0,100.0


# **Reporte_Discrepancias_ENSANUT_2023**

In [5]:
import pandas as pd
import os
import re

# ==========================================
# 1. CONFIGURACIÓN AUTOMÁTICA
# ==========================================
ANIO_ENCUESTA = 2023

# Rutas dinámicas basadas en el año
BASE_DIR = f'/content/ENSANUT_{ANIO_ENCUESTA}_Completa'
FILE_NAME = f'Diccionario_Validado_ENSANUT_{ANIO_ENCUESTA}.xlsx'
PATH_INPUT = os.path.join(BASE_DIR, FILE_NAME)
PATH_OUTPUT = f'Reporte_Discrepancias_ENSANUT_{ANIO_ENCUESTA}.xlsx'

# ==========================================
# 2. FUNCIONES DE LIMPIEZA Y ANÁLISIS
# ==========================================

def clean_val(v):
    """Normaliza valores: quita decimales .0, espacios y maneja nulos."""
    if pd.isna(v) or str(v).strip().lower() in ['nan', '']:
        return None
    v_str = str(v).strip()
    # Si es numérico con decimal .0 lo quitamos (ej. 1.0 -> 1)
    if re.match(r'^-?\d+\.0$', v_str):
        v_str = v_str[:-2]
    return v_str

def parse_catalog(catalog_str):
    """Extrae los códigos del formato '1.0=Etiqueta | 2.0=Etiqueta'."""
    if pd.isna(catalog_str) or str(catalog_str).strip().lower() == 'nan':
        return set()

    parts = str(catalog_str).split('|')
    codes = set()
    for p in parts:
        if '=' in p:
            code_part = p.split('=')[0].strip()
            cv = clean_val(code_part)
            if cv:
                codes.add(cv)
    return codes

def parse_matrix(matrix_str):
    """Extrae los códigos únicos de la matriz de valores encontrados."""
    if pd.isna(matrix_str) or str(matrix_str).strip().lower() == 'nan' or "(TRUNCADO)" in str(matrix_str):
        return set()

    parts = str(matrix_str).split(',')
    codes = set()
    for p in parts:
        cv = clean_val(p)
        if cv:
            codes.add(cv)
    return codes

# ==========================================
# 3. PROCESAMIENTO PRINCIPAL
# ==========================================

def ejecutar_analisis():
    if not os.path.exists(PATH_INPUT):
        print(f"Error: No se encontró el archivo en {PATH_INPUT}")
        return

    print(f"Leyendo archivo: {PATH_INPUT}...")
    # Se lee el archivo Excel (asegúrate de tener instalado openpyxl)
    df = pd.read_excel(PATH_INPUT)

    resultados = []
    print("Analizando variables...")

    for _, row in df.iterrows():
        matrix_vals = parse_matrix(row.get('VALORES_EN_MATRIZ'))
        catalog_vals = parse_catalog(row.get('CATÁLOGO_OFICIAL'))

        # Ignorar si no hay datos suficientes para comparar
        if not matrix_vals or not catalog_vals:
            continue

        # Hallar valores que están en los datos (matriz) pero NO en el diccionario
        undocumented = matrix_vals - catalog_vals

        if undocumented:
            resultados.append({
                'ANIO_ENCUESTA': ANIO_ENCUESTA,
                'COMPONENTE': row.get('COMPONENTE'),
                'ARCHIVO': row.get('ARCHIVO'),
                'VARIABLE': row.get('VARIABLE'),
                'ETIQUETA': row.get('ETIQUETA_PREGUNTA'),
                'VALORES_NO_DOCUMENTADOS': ", ".join(sorted(list(undocumented))),
                'VALORES_EN_MATRIZ': ", ".join(sorted(list(matrix_vals))),
                'CATALOGO_OFICIAL_RESUMEN': ", ".join(sorted(list(catalog_vals))),
                'CONTEO_EXTRAS': len(undocumented)
            })

    # Crear DataFrame con los hallazgos
    df_reporte = pd.DataFrame(resultados)

    # Guardar como Excel
    df_reporte.to_excel(PATH_OUTPUT, index=False)

    print("-" * 30)
    print(f"PROCESO FINALIZADO")
    print(f"Variables analizadas: {len(df)}")
    print(f"Variables con inconsistencias: {len(df_reporte)}")
    print(f"Archivo generado: {PATH_OUTPUT}")

if __name__ == "__main__":
    ejecutar_analisis()

Leyendo archivo: /content/ENSANUT_2023_Completa/Diccionario_Validado_ENSANUT_2023.xlsx...
Analizando variables...
------------------------------
PROCESO FINALIZADO
Variables analizadas: 3963
Variables con inconsistencias: 55
Archivo generado: Reporte_Discrepancias_ENSANUT_2023.xlsx


#PRUEBAS

In [6]:
import os

ruta_base = "/content/ENSANUT_2023_Completa"

for raiz, carpetas, archivos in os.walk(ruta_base):
    level = raiz.replace(ruta_base, '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f'{indent}{os.path.basename(raiz)}/')
    subindent = ' ' * 4 * (level + 1)
    for f in archivos:
        print(f'{subindent}{f}')

ENSANUT_2023_Completa/
    Diccionario_Validado_ENSANUT_2023.xlsx
    SALUD/
        Cuestionario de salud de niños (0 a 9 años)/
            SPSS/
                menores_ensanut2023_w_n.spss.spss.zip
                menores_ensanut2023_w_n.sav
        Cuestionario de salud de adultos (20 años o mas)/
            SPSS/
                adultos_ensanut2023_w_n.spss.spss.zip
                adultos_ensanut2023_w_n.sav
        Cuestionario de utilizadores de servicios de salud/
            SPSS/
                utilizadores_ensanut2023_w_n.spss.spss.zip
                utilizadores_ensanut2023_w_n.sav
        Cuestionario de salud de adolescentes (10 a 19 años)/
            SPSS/
                adolescentes_ensanut2023_w_n.spss.spss.zip
                adolescentes_ensanut2023_w_n.sav
        Cuestionario de hogar/
            Indice de bienestar/
                SPSS/
                    ENSANUT2023_ICB.spss.spss.zip
                    ENSANUT2023_ICB.sav
            Informacion sobre 

In [7]:
# 1. Instalar la dependencia necesaria (si estás en Colab/Jupyter)
!pip install pyreadstat

import pandas as pd

# 2. Definir la ruta del archivo
file_path = '/content/ENSANUT_2023_Completa/SALUD/Cuestionario de salud de adolescentes (10 a 19 años)/SPSS/adolescentes_ensanut2023_w_n.sav'

# 3. Cargar el archivo
# pd.read_spss devuelve un DataFrame.
# A veces es útil obtener también los metadatos (etiquetas de variables) usando pyreadstat directamente si pandas falla o si necesitas el diccionario de datos.
try:
    df_adolescentes = pd.read_spss(file_path,convert_categoricals =False)
    print("Archivo cargado exitosamente.")
    print(df_adolescentes.head())
except Exception as e:
    print(f"Error al cargar el archivo: {e}")

Archivo cargado exitosamente.
          FOLIO_INT        FOLIO_I maquina hora_ini_1 fecha_ini_1 hora_fin_1  \
0  2023_01001001_04  2023_01001001   MQ448   16:01:25  21/11/2023   16:02:07   
1  2023_01001003_05  2023_01001003   MQ448   13:17:11  22/11/2023   13:24:41   
2  2023_01001008_05  2023_01001008   MQ454   13:02:10  22/11/2023   13:13:02   
3  2023_01001010_05  2023_01001010   MQ454   15:26:50  21/11/2023   15:41:59   
4  2023_01001026_04  2023_01001026   MQ453   15:54:46  21/11/2023   16:16:14   

  fecha_fin_1  tiempo1  resultado_1 hora_ini_2  ...  hora_fin   fecha_fin  \
0  21/11/2023      1.0          2.0   16:17:17  ...  16:26:13  21/11/2023   
1  22/11/2023      7.0          2.0   13:25:38  ...  13:29:34  22/11/2023   
2  22/11/2023     11.0          1.0             ...  13:13:02  22/11/2023   
3  21/11/2023     15.0          1.0             ...  15:41:59  21/11/2023   
4  21/11/2023     22.0          1.0             ...  16:16:14  21/11/2023   

  completa  otroent  ageb 

In [8]:
df_adultos

,Grupo,Sección,Total Variables,Encontradas,Faltantes,Cobertura (%)
0,Adultos,I. SOBREPESO Y OBESIDAD,14,14,0,100.0
1,Adultos,II. SINTOMATOLOGÍA DEPRESIVA,15,15,0,100.0
2,Adultos,III. DIABETES MELLITUS,89,89,0,100.0
3,Adultos,IV. HIPERTENSIÓN ARTERIAL,38,38,0,100.0
4,Adultos,V. ENFERMEDAD CARDIOVASCULAR,4,4,0,100.0
5,Adultos,VI. ENF. RENAL/HIPERCOLESTEROLEMIA,15,15,0,100.0
6,Adultos,VII. ANTECEDENTES FAMILIARES,13,13,0,100.0
7,Adultos,VIII. SALUD SEXUAL Y REPRODUCTIVA,88,88,0,100.0
8,Adultos,IX. VACUNACION ADULTOS,82,82,0,100.0
9,Adultos,X. PROGRAMAS PREVENTIVOS,120,120,0,100.0


# Fin de automatización de SALUD se obtuvó un diccinario y acceso a las matrices.

# **AUTOMATIZACIÓN NUTRICION**

In [9]:
#@title ENSANUT FFQ -> Long + Wide (Volumen Semanal: p1*p2*p4)

!pip -q install pyreadstat pyarrow openpyxl



import os
from google.colab import files

# 1. Definir rutas
carpeta_a_comprimir = f"/content/ENSANUT_{ANIO_ENCUESTA}_Completa/NUTRICIÓN/Cuestionarios de frecuencia de consumo de alimentos"
nombre_archivo_salida = "nutricion_frecuencia.zip"

# 2. Comprimir usando comando de sistema
cmd = f'zip -r "{nombre_archivo_salida}" "{carpeta_a_comprimir}"'
os.system(cmd)

# 3. Descargar el archivo
print(f"Comprimiendo {carpeta_a_comprimir}...")


import re, zipfile, importlib.util, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
import pyreadstat

# ========= AJUSTA ESTO =========
INPUT = "/content/nutricion_frecuencia.zip"
OUT_DIR = Path("/content/ffq_out")
MODULE_PATH = "/content/frecuencia_de_alimentos_subcategorias.py"
DICT_PATH = f"/content/ENSANUT_{ANIO_ENCUESTA}_Completa/Diccionario_Validado_ENSANUT_{ANIO_ENCUESTA}.xlsx"
COMP_ALIMENTOS = "Frecuencia de consumo de alimentos - Alimentos"
# ===============================

OUT_DIR.mkdir(parents=True, exist_ok=True)

# 1. CATÁLOGOS Y UTILERÍAS
CATALOGO_OTRA_LECHE = {
    "7000": "Leche entera",
    "7001": "Leche descremada",
    "7002": "Leche en polvo descremada",
    "7003": "Leche evaporada descremada",
    "443": "Leche en polvo entera",
    "445": "Leche evaporada entera",
    "447": "Leche semidescremada",
    "1110": "Leche sin lactosa descremada",
    "1192": "Leche de soya",
    "7102": "Leche sin lactosa entera",
    "7103": "Leche bronca",
    "452": "Fórmula láctea para bebés de soya",
    "7048": "Atole de maíz con fórmula láctea para bebés en polvo",
    "7047": "Atole de maíz con leche entera",
    "7049": "Atole de maíz con leche bronca",
    "7050": "Atole de maíz con leche light",
    "7051": "Atole de maíz con leche en polvo descremada",
    "7052": "Atole de maíz con leche entera en polvo",
    "7054": "Atole de maíz con leche evaporada entera",
    "7055": "Atole de maíz con leche semidescremada",
    "7056": "Atole de maíz con leche sin lactosa",
    "7057": "Atole de maíz con leche sin lactosa entera",
    "7133": "Atole de maíz con leche evaporada descremada"
}

def resolve_input(input_path: str) -> Path:
    p = Path(input_path)
    if p.suffix.lower() == ".zip":
        work = Path("/content/_ffq_tmp_extract")
        work.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(p, "r") as z: z.extractall(work)
        return work
    return p

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    colmap = {
        "folio_int": "FOLIO_INT", "folioi": "FOLIO_I", "folio_i": "FOLIO_I",
        "alimento": "alimento", "por_est": "por_est", "esp_alim": "esp_alim",
        "pa1": "pa1", "pa2": "pa2", "pa3": "pa3", "pa4": "pa4"
    }
    df = df.rename(columns={c: colmap[c.lower()] for c in df.columns if c.lower() in colmap})
    for need in colmap.values():
        if need not in df.columns: df[need] = np.nan
    return df

def make_descripcion(df: pd.DataFrame) -> pd.DataFrame:
    alimento = df["alimento"].astype("string").fillna("").str.strip()
    por_est  = df["por_est"].astype("string").fillna("").str.strip()
    alimento_clean = alimento.str.replace(r"\s*opci[oó]n\s*[A-Za-z]\b", "", regex=True).str.replace(r"\s+", " ", regex=True).str.strip()
    esp_raw = df["esp_alim"].fillna(-1).astype(float).astype(int).astype(str).replace("-1", "")
    esp_texto = esp_raw.map(CATALOGO_OTRA_LECHE).fillna(esp_raw).astype("string")
    base = alimento_clean.where(~(alimento_clean.str.contains(r"\botra leche\b", case=False) & (esp_texto != "")), "Otra leche: " + esp_texto)
    df["descripcion"] = np.where((por_est != "") & (por_est.str.lower() != "nan"), base + " | Porción: " + por_est, base)
    return df

def coerce_freq_cols(df: pd.DataFrame) -> pd.DataFrame:
    for c in ["pa1","pa2","pa4"]: df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    return df

# ==============================================================================
# ✅ MODIFICACIÓN: Cálculo de volumen semanal (p1*p2*p4)
# ==============================================================================
def compute_features(df: pd.DataFrame) -> pd.DataFrame:
    # pa1 (días) * pa2 (veces) * pa4 (porciones) = Volumen Total Semanal
    df["volumen_semanal"] = df["pa1"] * df["pa2"] * df["pa4"]

    # Mantenemos las diarias solo para el formato LONG
    df["freq_dia_prom"] = (df["pa1"] * df["pa2"]) / 7.0
    df["porciones_dia_prom"] = df["volumen_semanal"] / 7.0
    return df

def archivos_por_componente(dict_path: str, componente_objetivo: str) -> set:
    dic = pd.read_excel(dict_path)
    dic.columns = dic.columns.str.strip().str.upper()
    d = dic[["ARCHIVO","COMPONENTE"]].dropna().copy()
    return set(d.loc[d["COMPONENTE"].eq(componente_objetivo), "ARCHIVO"].str.strip().str.lower().unique())

# --- EJECUCIÓN ---
root = resolve_input(INPUT)
allowed_files = archivos_por_componente(DICT_PATH, COMP_ALIMENTOS)
sav_files = sorted([p for p in root.rglob("*.sav") if p.name.strip().lower() in allowed_files])

dfs = []
for fp in sav_files:
    try:
        df, _ = pyreadstat.read_sav(str(fp), apply_value_formats=False)
        df = standardize_columns(df)
        df["module"] = "alimentos"
        df = make_descripcion(df)
        df = coerce_freq_cols(df)
        df = compute_features(df)

        # Solo registros con consumo real
        df = df[df["volumen_semanal"] > 0].copy()
        dfs.append(df)
        print(f"✅ {fp.name} procesado.")
    except Exception as e:
        print(f"❌ Error en {fp.name}: {e}")

ffq_long = pd.concat(dfs, ignore_index=True)

# ==============================================================================
# ✅ MODIFICACIÓN: WIDE usando pa1*pa2*pa4
# ==============================================================================
wide_freq_dia = ffq_long.pivot_table(
    index=["FOLIO_INT","FOLIO_I"],
    columns="descripcion",
    values="volumen_semanal", # <--- Aquí usamos la multiplicación directa
    aggfunc="sum",
    fill_value=0
)

# Guardar resultados
ffq_long.to_csv(OUT_DIR / "ffq_long.csv", index=False)
wide_freq_dia.to_csv(OUT_DIR / "ffq_wide_freq_dia_semanal.csv")

print("\n--- FINALIZADO ---")
print(f"Matriz WIDE generada con {wide_freq_dia.shape[1]} columnas de alimentos.")
print(f"Los valores representan el total de porciones consumidas por semana.")

Comprimiendo /content/ENSANUT_2023_Completa/NUTRICIÓN/Cuestionarios de frecuencia de consumo de alimentos...
✅ frec_ad_rec_w.sav procesado.
✅ frec_es_rec_w.sav procesado.
✅ frec_pr_rec_w.sav procesado.

--- FINALIZADO ---
Matriz WIDE generada con 274 columnas de alimentos.
Los valores representan el total de porciones consumidas por semana.


#**matriz de comida frecuencia porciones semanales**

In [12]:
wide_freq_dia

,descripcion,Agreagado a la leche: Azúcar | Porción: 1 cucharada cafetera copeteada (10g),Agreagado a la leche: Chocolate u otro saborizante | Porción: 1 cucharada cafetera copeteada (10g),Agregado a la leche: Azúcar | Porción: 1 cucharada cafetera copeteada (10g),Agregado a la leche: Chocolate u otro saborizante | Porción: 1 cucharada cafetera copeteada (10g),Agua sola | Porción: 1 vaso (240 ml),Agua sola | Porción: 1/2 vaso (120 ml),Aguacate | Porción: 1 rebanada o 1 pieza de criollo chico (33 g),Aguas de fruta natural con azúcar | Porción: 1 vaso (240 ml),Aguas de fruta natural con azúcar | Porción: 1/2 vaso (120 ml),Aguas de fruta natural sin azúcar | Porción: 1 vaso (240 ml),...,Yogur en vaso: Entero con frutas | Porción: 1 vaso típico de yogur (150 g),Yogur en vaso: Entero natural | Porción: 1 vaso típico de yogur (150g),Yogur para beber: Entero con frutas | Porción: 1 envase típico (230 g),Yogur para beber: Entero natural | Porción: 1 envase típico (230 g),Yogur para beber: a) Entero natural | Porción: 1 envase típico (230 g),Yogur para beber: b) Entero con fruta | Porción: 1 envase típico (230 g),"Yogur para beber: c) Bajo en grasa o light natural o con fruta (vitalinea, activia 0%, Siluette) | Porción: 1 envase típico (230 g)",Zanahoria | Porción: 1 pieza chica o ½ taza (50g),Zanahoria | Porción: 1 pieza chica o ½ taza (50g),Zanahoria | Porción: 1 pieza mediana o ½ taza (50g)
FOLIO_INT,FOLIO_I,,,,,,,,,,,,,,,,,,,,,
2023_01001001_04,2023_01001001,0.0,0.0,0.0,0.0,16.0,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023_01001002_01,2023_01001002,0.0,0.0,0.0,0.0,28.0,0.00,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2023_01001003_06,2023_01001003,0.0,0.0,0.0,0.0,14.0,0.00,2.0,3.5,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023_01001003_08,2023_01001003,0.0,0.0,0.0,0.0,0.0,18.62,10.5,0.0,7.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023_01001008_03,2023_01001008,0.0,0.0,0.0,0.0,10.5,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023_32024023_01,2023_32024023,0.0,0.0,0.0,0.0,28.0,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
2023_32024024_01,2023_32024024,0.0,0.0,0.0,0.0,35.0,0.00,4.0,28.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2023_32024028_01,2023_32024028,0.0,0.0,0.0,0.0,56.0,0.00,0.0,28.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0


In [ ]:
asdasdasd

In [13]:
# Para imprimir el conteo completo de todas las descripciones sin que se recorten los resultados:
print(ffq_long['descripcion'].value_counts().to_string())

# Si prefieres verlo como una lista de Python (solo los nombres de los alimentos):
print(ffq_long['descripcion'].value_counts().index.tolist())

descripcion
Agua sola | Porción: 1 vaso (240 ml)                                                                                                                                            2720
Frijoles preparados en casa: a) De la olla | Porción: ½ plato o ½ taza (50g)                                                                                                    2119
Refresco Normal | Porción: 1 vaso (240 ml)                                                                                                                                      2108
Arroz guisado | Porción: 1 taza o 1 plato (100g)                                                                                                                                1892
Huevo b) Frito, estrellado o revuelto | Porción: b) 1 pieza entera de huevo frito, estrellado o revuelto (55g)                                                                  1851
Pan dulce (excepto donas y churros) | Porción: 1 pieza (70g)                       

In [14]:
import pandas as pd
CATEGORY_TO_VARIABLES = {
    "WATER": [
        "Agua sola | Porción: 1 vaso (240 ml)",
        "Agua sola | Porción: 1/2 vaso (120 ml)"
    ],
    "BEV_NON_CALORIC": [
        "Refresco Dieta | Porción: 1 vaso (240 ml)",
        "Refresco Dieta | Porción: 1/2 vaso (120 ml)",
        "Café: a) Café sin azúcar | Porción: 1 taza (240 ml)",
        "Café: a) Café sin azúcar | Porción: 1/2 taza (120 ml)",
        "Té o infusión: a) Té sin azúcar | Porción: 1 taza (240 ml)",
        "Té o infusión: a) Té sin azúcar | Porción: 1/2 taza (120 ml)",
        "Bebidas o aguas de sabor industrializadas sin azúcar (incluyendo dietéticas como Clight, Be-light, etc.) | Porción: 1 vaso (240 ml)",
        "Bebidas o aguas de sabor industrializadas sin azúcar (incluyendo dietéticas como Clight, Be-light, etc.) | Porción: 1/2 vaso (120 ml)"
    ],
    "BEV_NATURAL_SUGAR": [
        "Jugos naturales sin azúcar | Porción: 1 vaso (240 ml)",
        "Jugos naturales sin azúcar | Porción: 1/2 vaso (120 ml)",
        "Aguas de fruta natural sin azúcar | Porción: 1 vaso (240 ml)",
        "Aguas de fruta natural sin azúcar | Porción: 1/2 vaso (120 ml)"
    ],
    "BEV_SSB": [
        "Refresco Normal | Porción: 1 vaso (240 ml)",
        "Refresco Normal | Porción: 1/2 vaso (120 ml)",
        "Aguas de fruta natural con azúcar | Porción: 1 vaso (240 ml)",
        "Aguas de fruta natural con azúcar | Porción: 1/2 vaso (120 ml)",
        "Jugos naturales con azúcar | Porción: 1 vaso (240 ml)",
        "Jugos naturales con azúcar | Porción: 1/2 vaso (120 ml)",
        "Néctares de frutas o pulpa de frutas industrializados con azúcar (boing, jumex) | Porción: 1 vaso (240 ml)",
        "Néctares de frutas o pulpa de frutas industrializados con azúcar (boing, jumex) | Porción: 1/2 vaso (120 ml)",
        "Bebidas o aguas de sabor industrializadas con azúcar (frutsi, bonafina.) | Porción: 1 vaso (240 ml)",
        "Bebidas o aguas de sabor industrializadas con azúcar (frutsi, bonafina.) | Porción: 1/2 vaso (120 ml)"
    ],
    "ALCOHOL": [
        "Bebidas alcohólicas | Porción: 1 vaso (240 ml) de cerveza, vino, pulque, cuba o copa sólo con tequila, mezcal u otro"
    ],
    "AROMATICS": [
        "Tomate verde y jitomate por ejemplo en salsas, tacos, o guisados (molido o entero)",
        "Limón por ejemplo en ensaladas, caldos, o carnes",
        "Cebolla por ejemplo en salsas, o caldillos (molida o entera)",
        "Jitomate | Porción: ½ pieza chica (30g) en ensalada",
        "Chiles frescos por ejemplo en salsas, tacos, guisados (molido o entero)",
        "Cebolla por ejemplo en ensaladas, antojitos, o comida rápida | Porción: 1 cucharada sopera o 3 rodajas (10g)",
        "Chile seco por ejemplo en salsas, tacos, guisados (molido o entero)",
        "Chiles envasados o enlatados, por ejemplo en el sándwich, torta, guisados."
    ],
    "VEG": [
        "Lechuga | Porción: ½ taza o 1 hoja (30g)",
        "Pepino | Porción: 1/2 pieza grande (150g)",
        "Calabacita | Porción: ½ pieza mediana (50g)",
        "Zanahoria | Porción: 1 pieza mediana o ½ taza (50g)",
        "Zanahoria | Porción: 1 pieza chica o ½ taza (50g)",
        "Zanahoria | Porción: 1  pieza  chica o ½ taza (50g)",
        "Nopales | Porción: 1 pieza grande (100g) o 3/4 taza",
        "Nopales | Porción: 1 pieza mediana (70g) o 1/2 taza",
        "Hojas Verdes (acelgas, espinacas, quelites) | Porción: ½ plato (85g) cocidas o 1 plato crudas",
        "Chayote | Porción: ¼ pieza chica (50g) o 1/3 taza",
        "Brócoli o coliflor | Porción: ¼ taza (35g)",
        "Brócoli o coliflor | Porción: ¼  taza (35g)",
        "Ejotes | Porción: ¼ taza o 5 piezas (30g)",
        "Col | Porción: ¼ taza (35 g)",
        "Jícama | Porción: ½ pieza mediana (163g)",
        "Jícama | Porción: 1/3 pieza mediana o 3/4 taza (100g)",
        "Chile poblano | Porción: Una pieza mediana o 1/3 taza \xa0(80g)",
        "Chile poblano | Porción: 1/2 pieza mediana \xa0(40g)",
        "Calabacita | Porción: ½  pieza  mediana  (50g)",
        "Verduras envasadas como chícharo, zanahoria, champiñones y ejotes. | Porción: 1/3 taza o 1 lata pequeña",
        "Verduras envasadas como chícharo, zanahoria, champiñones y ejotes. | Porción: 1/2 lata pequeña",
        "Verduras congeladas como chícharo, zanahoria, brócoli, coliflor, ejotes | Porción: 1/3 taza",
        "Verduras congeladas como chícharo, zanahoria, brócoli, coliflor, ejotes | Porción: 1/4 taza"
    ],
    "FRUIT_FRESH": [
        "Plátano | Porción: 1 pieza mediana (176g)  o 2 plátanos dominicos medianos",
        "Plátano | Porción: 1 pieza mediana (176g) o 2 plátanos dominicos medianos",
        "Manzana o pera | Porción: 1 pieza mediana (140g)",
        "Manzana o pera | Porción: 1/2 pieza grande (108g)",
        "Manzana o pera | Porción: 1/2 pieza mediana (70g)",
        "Uvas | Porción: 10 piezas (60g)",
        "Naranja o mandarina | Porción: 1 pieza grande (206g)",
        "Naranja o mandarina | Porción: 1 pieza mediana(145g)",
        "Melón o sandía | Porción: 1 rebanada o 3/4 taza (115g)",
        "Melón o sandía | Porción: 1 rebanada mediana o 3/4  taza (115g)",
        "Melón o sandía | Porción: 1 rebanada mediana o 3/4 taza (115g)",
        "Plátano | Porción: 1 pieza chica (116g)  o 2 plátanos dominicos chicos",
        "Papaya | Porción: 1 rebanada mediana (100g) o ½ taza",
        "Papaya | Porción: 1/2 taza o 1 rebanada chica (70g)",
        "Guayaba | Porción: 1 pieza mediana(75g)",
        "Guayaba | Porción: 1 pieza mediana (75g)",
        "Guayaba | Porción: 1 pieza chica (50g)",
        "Fresa | Porción: 1 taza (140 g)",
        "Fresa | Porción: 1/3 taza o 3 piezas medianas (50 g)",
        "Naranja o mandarina | Porción: 1 pieza chica (109g)",
        "Piña | Porción: 1 rebanada mediana (150 g)",
        "Piña | Porción: 1/2  rebanada mediana (75 g)",
        "Piña | Porción: 1/2 rebanada mediana (75 g)",
        "Mango | Porción: 1 pieza mediana (185g)",
        "Mango | Porción: 1/2 pieza chica (62g)",
        "Durazno/melocotón | Porción: 1 pieza mediana (55g)",
        "Toronja | Porción: 1 pieza chica (270 g)",
        "Toronja | Porción: 1/2 pieza chica (135g)",
        "Toronja | Porción: 1/2  pieza chica (135g)"
    ],
    "FRUIT_PROC": [
        "Frutas cristalizadas o secas | Porción: ¼ taza (25g)",
        "Frutas cristalizadas o secas | Porción: ¼  taza (25g)",
        "Frutas en almíbar | Porción: ½ taza (80g)",
        "Frutas en almíbar | Porción: ½  taza (80g)"
    ],
    "LEGUMES": [
        "Frijoles preparados en casa: a) De la olla | Porción: ½ plato o ½ taza (50g)",
        "Frijoles preparados en casa: b) Refritos | Porción: ½ plato o ½ taza (50g)",
        "Lenteja, garbanzo, haba amarilla o alubia | Porción: 1 plato o 1 taza (100g)",
        "Lenteja, garbanzo, haba amarilla o alubia | Porción: 1/2 plato o 1/2 taza (50g)",
        "Haba, garbanzo o lentejas como botana | Porción: 1 cucharada sopera (10 g)",
        "Frijoles envasados o de lata: a) De la olla | Porción: ½ plato o ½ taza (50g)",
        "Frijoles envasados o de lata: a) De la olla | Porción: ½ plato o ½ taza  (50g)",
        "Caldo de frijol | Porción: 1/2 plato o 1/2 taza (50g)",
        "Frijoles envasados o de lata: b) Refritos | Porción: ½ plato o ½ taza (50g)",
        "Frijoles envasados o de lata: b) Refritos | Porción: ½ plato o ½ taza  (50g)"
    ],
    "NUTS_AVO": [
        "Aguacate | Porción: 1 rebanada o 1 pieza de criollo chico (33 g)",
        "Nuez, almendra, avellana, cacahuates, semilla de calabaza (pepita) o de girasol, pistache, piñón, etc. | Porción: 1 cucharada sopera (10 g)"
    ],
    "FATS_ADDED": [
        "Mayonesa | Porción: 1 cucharada sopera (10g)",
        "Crema | Porción: 1 cucharada sopera (10g)",
        "Mantequilla | Porción: 1 cucharada sopera (10g)",
        "Margarina | Porción: 1 cucharada sopera (10g)",
        "Manteca animal (cerdo o pollo) | Porción: 1 cucharada sopera (10g)",
        "Manteca vegetal | Porción: 1 cucharada sopera (10g)",
        "Café: d) Sustituto de crema agregada al café | Porción: 1 cucharada sopera",
        "Café: d) Sustituto de crema agregada al café | Porción: 1 cucharada sopera (10g)"
    ],
    "SUGAR_ADDED": [
        "Café: b) Azúcar agregada al café | Porción: 1 cucharada cafetera copeteada (10g)",
        "Té o infusión: b) Azúcar agregada al té | Porción: 1 cucharada cafetera copeteada (10g)",
        "Agregado a la leche: Azúcar | Porción: 1 cucharada cafetera copeteada (10g)",
        "Azúcar (a parte de la agregada a las bebidas, leche, té, café, agua de frutas) por ejemplo en fresas o plátanos con crema | Porción: 1 cucharada sopera (10g)",
        "Agreagado a la leche: Azúcar | Porción: 1 cucharada cafetera copeteada (10g)"
    ],
    "DAIRY_UNSWEET": [
        "Otra leche: Leche entera | Porción: 1 vaso (240 ml)",
        "Otra leche: Leche entera | Porción: 1 vaso  (240 ml)",
        "Quesos madurados (chihuahua, manchego, gouda, etc.) | Porción: 1 rebanada (30 g)",
        "Queso panela o fresco o cottage | Porción: 1 rebanada o 2 cuaharadas soperas(30 g)",
        "Queso panela o fresco o cottage | Porción: 1 rebanada o 2 cucharadas soperas (30 g)",
        "Leche Liconsa | Porción: 1 vaso (240 ml)",
        "Leche Liconsa | Porción: 1 vaso  (240 ml)",
        "Leche Liconsa | Porción: 1 vaso (240ml)",
        "Otra leche: Leche sin lactosa entera | Porción: 1 vaso (240 ml)",
        "Otra leche: Leche sin lactosa entera | Porción: 1 vaso  (240 ml)",
        "Café: c) Leche agregada al café (aparte de la reportada en el apartado de “Productos lácteos”) | Porción: 1 taza (240 ml): (especificar tipo de leche)",
        "Café: c) Leche agregada al café (aparte de la reportada en el apartado de “Productos lácteos”) | Porción: 1/2 taza (120 ml): (especificar tipo de leche)",
        "Otra leche: Leche bronca | Porción: 1 vaso (240 ml)",
        "Yogur de vaso: a) Entero natural | Porción: 1 vaso típico de yogur (150g)",
        "Yogur para beber: a) Entero natural | Porción: 1 envase típico (230 g)",
        'Yogur para beber: Entero natural | Porción: 1 envase típico (230 g)', #############
        "Yogur en vaso: Entero natural | Porción: 1 vaso típico de yogur (150g)",
        "Otra leche: Leche sin lactosa descremada | Porción: 1 vaso (240 ml)",
        "Otra leche: Leche sin lactosa descremada | Porción: 1 vaso  (240 ml)",
        "Otra leche: Leche descremada | Porción: 1 vaso (240 ml)",
        "Otra leche: Leche descremada | Porción: 1 vaso  (240 ml)",
        "Otra leche: Leche semidescremada | Porción: 1 vaso (240 ml)",
        "Otra leche: Leche semidescremada | Porción: 1 vaso  (240 ml)",
        "Otra leche: Leche evaporada entera | Porción: 1 vaso (240 ml)",
        "Otra leche: Leche evaporada entera | Porción: 1 vaso  (240 ml)",
        "Otra leche: Leche en polvo entera | Porción: 1 vaso (240 ml)",
        "Otra leche: Leche en polvo entera | Porción: 1 vaso  (240 ml)",
        "Otra leche: Leche en polvo descremada | Porción: 1 vaso (240 ml)",
        "Otra leche: Leche en polvo descremada | Porción: 1 vaso  (240 ml)",
        "Otra leche: Leche de soya | Porción: 1 vaso (240 ml)",
        "Otra leche: Leche de soya | Porción: 1 vaso  (240 ml)",
        "Otra leche: Leche evaporada descremada | Porción: 1 vaso (240 ml)",
        "Otra leche: Leche evaporada descremada | Porción: 1 vaso  (240 ml)"
    ],
    "DAIRY_SWEET": [
        "Yakult o similares | Porción: 1 envase (80ml)",
        "Danonino o similar | Porción: 1 envase (45g)",
        "Agregado a la leche: Chocolate u otro saborizante | Porción: 1 cucharada cafetera copeteada (10g)",
        "Yogur para beber: Entero con frutas | Porción: 1 envase típico (230 g)",
        "Agreagado a la leche: Chocolate u otro saborizante | Porción: 1 cucharada cafetera copeteada (10g)",
        "Leche preparada de sabor (chocolate u otro sabor) | Porción: 1 vaso (240 ml)",
        "Leche preparada de sabor (chocolate u otro sabor) | Porción: 1 vaso  (240 ml)",
        "Yogur para beber: b) Entero con fruta | Porción: 1 envase típico (230 g)",
        "Yogur de vaso: b) Entero con frutas | Porción: 1 vaso típico de yogur (150 g)",
        "Yogur en vaso: Entero con frutas | Porción: 1 vaso típico de yogur (150 g)",
        "Yogur de vaso: c) Bajo en grasa o light natural o con fruta (vitalinea, alpura light, lala light, etc.) | Porción: 1 vaso típico de yogur (150g)",
        "Yogur para beber: c) Bajo en grasa o light natural o con fruta (vitalinea, activia 0%, Siluette) | Porción: 1 envase típico (230 g)",
        "Yogur en vaso: Bajo en grasa o light natural o con fruta (vitalínea, alpura light, lala light, etcétera) | Porción: 1 vaso típico de yogur (150g)"
    ],
    "PROT_UNPROC": [
        "Pollo a) Pierna, muslo o pechuga | Porción: a) 1 pieza (pierna, muslo) o ½ pieza de pechuga chica (90g)",
        "Huevo b) Frito, estrellado o revuelto | Porción: b) 1 pieza entera de huevo frito, estrellado o revuelto (55g)",
        "Carne de puerco | Porción: 1 bistec mediano (90g)",
        "Carne de res | Porción: 1 bistec mediano (90g)",
        "Huevo: b) Frito, estrellado o revuelto | Porción: b) 1 pieza entera de huevo frito, estrellado o revuelto (55g)",
        "Pollo: a) Pierna, muslo o pechuga | Porción: a) 1 pieza (pierna, muslo) o ½ pieza de pechuga chica (90g)",
        "Huevo a) Tibio o cocido | Porción: a) 1 pieza entera de huevo tibio o cocido (62g)",
        "Atún y sardina (en tomate, agua o aceite) | Porción: ¼ lata o 40g",
        "Pollo b) Ala, patas | Porción: b) 1 pieza de ala, 2 piezas de patas (70g)",
        "Carne de puerco | Porción: 1 bistec chico (55g)",
        "Pescado fresco | Porción: 1 filete mediano o mojarra chica (90 g)",
        "Carne de res | Porción: 1 bistec chico (55g)",
        "Algún marisco (camarón, ostiones, etc.) | Porción: 1 plato (100g)",
        "Carne de res | Porción: 1/2 bistec mediano (45g)",
        "Carne de puerco | Porción: 1/2 bistec mediano (45g)",
        "Huevo: a) Tibio o cocido | Porción: a) 1 pieza entera de huevo tibio o cocido (62g)",
        "Pollo c) Higadito o molleja | Porción: c) 1 pieza de higadito o molleja (30g)",
        "Pollo: b) Alas, patas | Porción: b) 1 pieza de ala, 2 piezas de patas (70g)",
        "Algún marisco (camarón, ostiones, etc.) | Porción: 1/2 plato (50g)",
        "Pescado fresco | Porción: 1/2  filete mediano (45g)",
        "Pollo b) Ala, patas | Porción: b) 1pieza de ala, 2 piezas de patas (70g)",
        "Pescado fresco | Porción: 1/2 filete mediano (45g)",
        "Atún y sardina (en tomate, aguo o aceite) | Porción: ¼ lata o 40g",
        "Pollo: c) Higado o mollejas | Porción: c) 1 pieza de higadito o molleja  (30g)"
    ],
    "PROT_PROC": [
        "Longaniza o chorizo | Porción: ½ trozo (30g)",
        "Salchicha de puerco, pavo o combinado, jamón de puerco o pavo o mortadela (a parte de en torta, sándwich o hot dog) | Porción: 1 pieza de salchicha o 1 reb. de jamón (30g)",
        "Salchicha de puerco, pavo o combinado, jamón de puerco o pavo o mortadela (a parte de en torta, sándwich o hot dog) | Porción: 1 pieza de salchicha  o 1 reb. de jamón (30g)",
        "Carne de res seca (machaca) | Porción: 1 plato (80g)",
        "Carne de res seca (machaca) | Porción: 1/2 plato (40g)",
        "Pescado seco (charalitos, bacalao) | Porción: 1 plato (80)",
        "Pescado seco (charalitos, bacalao) | Porción: 1/2 plato (40g)"
    ],
    "WHOLE_GRAINS": [
        "Avena en hojuelas, amaranto natural o tostado | Porción: 1/3 de taza (30 g)",
        "Pan integral | Porción: 2 rebanadas o 1 bolillo (70g)",
        "Galletas integrales | Porción: 4 piezas (20 g)",
        "Cereal d) Básico (Corn Flakes, arroz inflado sin sabor) | Porción: 1 taza (seco 30 g)",
        "Pan integral | Porción: 1 rebanadas o 1/2 bolillo (45g)",
        "Cereal b) Light/cuidado de la figura (Special K) | Porción: 1 taza (seco 30 g)",
        "Cereal g) Fibra (All Bran) | Porción: 1 taza (seco 30 g)",
        "Cereal de caja: b) Light/cuidado de la figura (Special K) | Porción: 1 taza  (seco 30 g)",
        "Cereal de caja: g) Fibra (All Bran) | Porción: 1 taza  (seco 30 g)",
        "Cereal de caja: d) Básico (Corn Flakes, arroz inflado sin sabor) | Porción: 1 taza  (seco 30 g)"
    ],
    "CEREAL_SWEET": [
        "Cereal a) Chocolate (Chocozucaritas, chocokrispis) | Porción: 1 taza (seco 30 g)",
        "Cereal c) Hojuela endulzada (Zucaritas) | Porción: 1 taza (seco 30 g)",
        "Cereal de caja: c) Hojuela endulzada (Zucaritas) | Porción: 1 taza  (seco 30 g)",
        "Cereal de caja: a) Chocolate (Chocozucaritas, chocokrispis) | Porción: 1 taza  (seco 30 g)",
        "Cereal f) Sabor a frutas (Froot loops) | Porción: 1 taza (seco 30 g)",
        "Cereal de caja: f) Sabor a frutas (Froot loops) | Porción: 1 taza  (seco 30 g)",
        "Cereal e) Variedades (Apple jacks, honey smacks, corn pops) | Porción: 1 taza (seco 30 g)",
        "Cereal de caja: e) Variedades (Apple jacks, honey smacks, corn pops) | Porción: 1 taza  (seco 30 g)",
        "Cereal i) Multi ingredientes (Extra) | Porción: 1 taza (seco 30 g)",
        "Cereal h) Especialidades (Crusli) | Porción: 1 taza (seco 30 g)",
        "Cereal de caja: h) Especialidades (Crusli) | Porción: 1 taza  (seco 30 g)",
        "Cereal de caja: i) Multi ingredientes (Extra) | Porción: 1 taza  (seco 30 g)"
    ],
    "STARCHES": [
        "Arroz guisado | Porción: 1 taza o 1 plato (100g)",
        "Sopa de pasta a) Sopa caldosa | Porción: a) 1 plato o 1 taza sopa caldosa (100g)",
        "Pan blanco | Porción: 2 rebanadas o 1 bolillo (70g)",
        "Sopa de pasta: a) Sopa caldosa | Porción: a) 1/2 plato o 1/2 taza sopa caldosa (50g)",
        "Arroz guisado | Porción: 1/2 taza o 1/2 plato (50g)",
        "Pozole (todos tipos) | Porción: 1 plato (100 g)",
        "Sopa de pasta a) Sopa caldosa | Porción: a) 1/2 plato o 1/2 taza sopa caldosa (50g)",
        "Sopa de pasta b) Sopa seca | Porción: b) 1 plato sopa seca (100g)",
        "Sopa de pasta: b) Sopa seca | Porción: b) 1/2 plato sopa seca (50g)",
        "Atole de maíz a) Atole con agua o pozol | Porción: 1 taza (240ml)",
        "Atole de maíz b) Atole con leche (aparte de la reportada en el apartado de “Productos lácteos”) | Porción: 1 taza (240 ml): (especificar tipo de leche)",
        "Papas: a) Cocida | Porción: a) ½ pieza mediana cocida (40g)",
        "Sopa de pasta b) Sopa seca | Porción: b) 1/2 plato sopa seca (50g)",
        "Tamal (todos tipos) | Porción: 1 pieza (200 g)",
        "Pan blanco | Porción: 1 rebanadas o 1/2 bolillo (45g)",
        "Tamal (todos tipos) | Porción: 1/2 pieza (100 g)",
        "Papas a) Cocida | Porción: a) ½ pieza mediana cocida (40g)",
        "Atole de maíz b) Atole con leche (aparte de la reportada en el apartado de “Productos lácteos”) | Porción: 1/2 taza (120 ml): (especificar tipo de leche)",
        "Pozole (todos tipos) | Porción: 1/2 plato (50 g)",
        "Atole de maíz a) Atole con agua o pozol | Porción: 1/2 taza (120ml)",
        "Elote | Porción: ½ pieza chica (50g)",
        "Tamal (todos tipos) | Porción: 1 pza (200 g)"
    ],
    "ANTOJ_NOFRY": [
        "Antojitos con vegetales como sopes, quesadillas, tlacoyos, gorditas, y enchiladas (NO TACOS): a) Sin freír | Porción: 100 g",
        "Antojitos con res, cerdo pollo, vísceras, etc como tacos, quesadillas, tlacoyos, enchiladas, gorditas: a) Sin freír | Porción: 100 g",
        "Frijoles envasados o de lata: a) De la olla | Porción: ½ plato o ½ taza (50g)"
    ],
    "FAST_FOOD": [
        "Torta o sándwich con pan blanco | Porción: 1 pieza mediana (130g)",
        "Pizza | Porción: 1 rebanada chica (92g)",
        "Torta o sándwich con pan integral | Porción: 1 pieza mediana (130g)",
        "Hamburguesa | Porción: 1 pieza mediana (240g)",
        "Hot dog | Porción: 1 pieza mediana (110g)"
    ],
    "FRIED": [
        "Papas b) Frita o tortita de papa | Porción: b) 1/2 pieza mediana frita o ½ tortita de papa (40g)",
        "Antojitos con res, cerdo pollo, vísceras, etc como tacos, quesadillas, tlacoyos, enchiladas, gorditas: b) Fritos | Porción: 100 g",
        "Antojitos con vegetales como sopes, quesadillas, tlacoyos, gorditas, y enchiladas (NO TACOS): b) Fritos | Porción: 100 g",
        "Papas: b) Frita o tortita de papa | Porción: b) 1/2 pieza mediana frita o ½ tortita de papa (40g)",
        "Tortitas de verduras capeadas | Porción: 1 pieza (72g)",
        "Plátano frito | Porción: ½ pieza mediana (113g)",
        "Tortitas de verduras capeadas | Porción: 1 pieza mediana (72g)"
    ],
    "SOUPS_HOME": [
        "Sopa o caldo con verduras | Porción: 1 plato (240 ml)",
        "Caldo de pollo, res o verduras (sólo caldo) | Porción: 1 taza (240 ml)",
        "Sopa o caldo con verduras | Porción: 1/2 plato (120ml)",
        "Caldo de pollo, res o verduras (sólo caldo) | Porción: 1/2 taza (120 ml)",
        "Crema de verduras | Porción: 1 plato (240 ml)",
        "Crema de verduras | Porción: 1/2 plato (120ml)"
    ],
    "SOUP_INST": [
        "Sopas instantáneas | Porción: 1 vaso (64g)",
        "Sopas instantáneas | Porción: 1/2 vaso (32g)"
    ],
    "SNACK_SALTY": [
        "Frituras (todos tipos, incluyendo cacahuates japoneses) | Porción: 1 paquete individual o bolsa chica (35g)",
        "Palomitas de maíz caseras, de microondas o del cine (Todos tipos, excepto acarameladas) | Porción: 1 bolsa mediana (100g) o 10 tazas de palomitas caseras",
        "Galletas saladas | Porción: 4 piezas (20g)",
        "Palomitas de maíz caseras, de microondas o del cine (Todos tipos, excepto acarameladas) | Porción: 1/2 bolsa mediana (50g)  o 5 tazas de palomitas caseras",
        "Galletas saladas | Porción: 4 piezas  (20g)"
    ],
    "SWEETS": [
        "Pan dulce (excepto donas y churros) | Porción: 1 pieza (70g)",
        "Galletas dulces (todos tipos) | Porción: 2 piezas (32g)",
        "Dulce (caramelos, paletas) | Porción: 1 pieza (30g)",
        "Gelatina, flan | Porción: 1 pieza o rebanada (125g)",
        "Chocolate | Porción: 1 trozo o 1 cucharada sopera (10g)",
        "Pastel o pay | Porción: 1 rebanada mediana(125 g)",
        "Helado, nieves y paletas de agua | Porción: 1 pza o 1 bola (80g)",
        "Donas y churros de panadería | Porción: 1 pieza (70 g)",
        "Helado y paletas de leche | Porción: 1 pza o 1 bola (80g)",
        "Dulces enchilados (miguelitos, tamarindos) | Porción: 1 pieza (30 g)",
        "Chocolate | Porción: 1  trozo o 1 cucharada sopera (10g)",
        "Pastelillos y donas industrializadas | Porción: 1 pieza (70g)",
        "Paletas y dulces de malvavisco (paleta payaso, bubu-lu-bu) | Porción: 2 piezas pequeñas o 1 pieza grande (40g)",
        "Pastel o pay | Porción: 1 rebanada mediana (125 g)",
        "Barras de cereal | Porción: 1 pieza (25g)",
        "Pastelillos y donas industrializadas | Porción: 1 pieza  (70g)"
    ],
    "INFANT": [
        "Leche materna | Porción: 1 tetada",
        "Otra leche: Fórmula láctea para bebés de soya | Porción: 1 vaso (240 ml)",
        "Otra leche: Fórmula láctea para bebés de soya | Porción: 1 vaso  (240 ml)"
    ],
    "COND_SOD": [
        "Sal o condimento con sal agregada a sus alimentos | Porción: (sal de mesa, sal con ajo, sal con cebolla)",
        "Salsas y aderezos agregados a sus alimentos: a) Cátsup",
        "Salsas y aderezos agregados a sus alimentos: b) Salsa picante para botana agregada a sus alimentos",
        "Sal o condimento con sal agregada a sus alimentos",
        "Salsas y aderezos agregados a sus alimentos: c) Salsa de soya, salsa inglesa o sazonadores líquidos agregados a sus alimentos"
    ]
}
# 3. Transformación: "Invertimos" el diccionario para búsqueda rápida (Mapping)
# Esto convierte {CATEGORIA: [item1, item2]} en {item1: CATEGORIA, item2: CATEGORIA}
ITEM_MAP = {}
for group, items_list in CATEGORY_TO_VARIABLES.items():
    for item in items_list:
        ITEM_MAP[item] = group


# 4. Función de asignación optimizada (búsqueda directa en diccionario)
def assign_group_fast(desc: str):
    """
    Busca la descripción exacta en el mapa pre-generado.
    Devuelve: (código, nombre_grupo, descripción_original)
    """
    # Intentamos buscar directo. Si hay espacios extra en tus datos, usa desc.strip()
    code = ITEM_MAP.get(desc, "UNCLASS")

    label = CATEGORY_TO_VARIABLES.get(code, "Sin clasificar")

    return code, label, desc


# -----------------------------
# 5) Uso con tu DataFrame
# -----------------------------
if 'ffq_long' in globals():
    # Obtenemos items únicos para no procesar repetidos innecesariamente
    unique_items = ffq_long['descripcion'].unique()

    def build_mapping_df(items):
        # Creamos DataFrame temporal
        df_temp = pd.DataFrame({"descripcion": items})

        # Aplicamos la función optimizada
        df_temp[["grupo", "grupo_nombre", "descripcion_base"]] = df_temp["descripcion"].apply(
            lambda x: pd.Series(assign_group_fast(x))
        )
        return df_temp

    # Generamos el mapeo
    df_map = build_mapping_df(unique_items)

    # Unimos el mapeo de vuelta al dataframe grande (ffq_long)
    # Esto es mucho más rápido que aplicar la función a 1 millón de filas
    ffq_long_classified = ffq_long.merge(df_map[['descripcion', 'grupo', 'grupo_nombre']], on='descripcion', how='left')

    # Imprimimos resultados de verificación
    print("--- CONTEO POR GRUPOS ---")
    print(ffq_long_classified["grupo_nombre"].value_counts().head(20))

    print("\n--- SIN CLASIFICAR ---")
    unclass = ffq_long_classified[ffq_long_classified["grupo"]=="UNCLASS"]
    print(f"Total filas sin clasificar: {len(unclass)}")
    if len(unclass) > 0:
        print("Ejemplos sin clasificar:")
        print(unclass["descripcion"].unique()[:10])

else:
    print("⚠️ 'ffq_long' no está definido. Ejecuta primero la carga de datos.")

--- CONTEO POR GRUPOS ---
grupo_nombre
[Pollo a) Pierna, muslo o pechuga | Porción: a) 1 pieza (pierna, muslo) o ½ pieza de pechuga chica (90g), Huevo b) Frito, estrellado o revuelto | Porción: b) 1 pieza entera de huevo frito, estrellado o revuelto (55g), Carne de puerco | Porción: 1 bistec mediano (90g), Carne de res | Porción: 1 bistec mediano (90g), Huevo: b) Frito, estrellado o revuelto | Porción: b) 1 pieza entera de huevo frito, estrellado o revuelto (55g), Pollo: a) Pierna, muslo o pechuga | Porción: a) 1 pieza (pierna, muslo) o ½ pieza de pechuga chica (90g), Huevo a) Tibio o cocido | Porción: a) 1 pieza entera de huevo tibio o cocido (62g), Atún y sardina (en tomate, agua o aceite) | Porción: ¼ lata o 40g, Pollo b) Ala, patas | Porción: b) 1 pieza de ala, 2 piezas de patas (70g), Carne de puerco | Porción: 1 bistec chico (55g), Pescado fresco | Porción: 1 filete mediano o mojarra chica (90 g), Carne de res | Porción: 1 bistec chico (55g), Algún marisco (camarón, ostiones, etc

In [16]:
# ==============================================================================
# ✅ AGRUPAMIENTO FINAL: Suma de porciones por Categoría
# ==============================================================================

# 1. Crear el mapa inverso { 'Descripción larga' : 'Categoría' }
# Usamos tu diccionario CATEGORY_TO_VARIABLES
item_to_category = {}
for category, descriptions in CATEGORY_TO_VARIABLES.items():
    for desc in descriptions:
        item_to_category[desc] = category

# 2. Re-mapear las columnas y sumar (axis=1 indica que agrupamos columnas)
# Usamos .get(x, 'UNCLASS') para capturar lo que no esté en el diccionario
wide_grouped = wide_freq_dia.groupby(
    wide_freq_dia.columns.map(lambda x: item_to_category.get(x, 'UNCLASS')),
    axis=1
).sum()

# 3. Ordenar columnas alfabéticamente para facilitar la lectura
wide_grouped = wide_grouped.reindex(sorted(wide_grouped.columns), axis=1)

# --- RESULTADOS ---
print(f"✅ Matriz agrupada generada: {wide_grouped.shape}")
print(f"Categorías encontradas: {wide_grouped.columns.tolist()}")

# Mostrar los primeros 5 folios con sus sumas por grupo
display(wide_grouped.head())

# Guardar en CSV para tu análisis
wide_grouped.to_csv(OUT_DIR / "ffq_final_grouped_sums.csv")



✅ Matriz agrupada generada: (3356, 28)
Categorías encontradas: ['ALCOHOL', 'ANTOJ_NOFRY', 'AROMATICS', 'BEV_NATURAL_SUGAR', 'BEV_NON_CALORIC', 'BEV_SSB', 'CEREAL_SWEET', 'DAIRY_SWEET', 'DAIRY_UNSWEET', 'FAST_FOOD', 'FATS_ADDED', 'FRIED', 'FRUIT_FRESH', 'FRUIT_PROC', 'INFANT', 'LEGUMES', 'NUTS_AVO', 'PROT_PROC', 'PROT_UNPROC', 'SNACK_SALTY', 'SOUPS_HOME', 'SOUP_INST', 'STARCHES', 'SUGAR_ADDED', 'SWEETS', 'VEG', 'WATER', 'WHOLE_GRAINS']


/tmp/ipython-input-1379554673.py:14: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  wide_grouped = wide_freq_dia.groupby(


,descripcion,ALCOHOL,ANTOJ_NOFRY,AROMATICS,BEV_NATURAL_SUGAR,BEV_NON_CALORIC,BEV_SSB,CEREAL_SWEET,DAIRY_SWEET,DAIRY_UNSWEET,FAST_FOOD,...,PROT_UNPROC,SNACK_SALTY,SOUPS_HOME,SOUP_INST,STARCHES,SUGAR_ADDED,SWEETS,VEG,WATER,WHOLE_GRAINS
FOLIO_INT,FOLIO_I,,,,,,,,,,,,,,,,,,,,,
2023_01001001_04,2023_01001001,0.0,1.0,0.0,0.0,0.0,28.0,1.0,0.0,6.0,4.0,...,13.0,6.58,1.0,0.5,3.0,21.0,9.0,0.0,16.00,0.0
2023_01001002_01,2023_01001002,4.0,0.0,2.0,0.5,0.0,14.0,0.0,0.0,5.0,0.0,...,2.0,0.00,0.0,0.0,1.0,7.5,3.0,18.0,28.00,0.0
2023_01001003_06,2023_01001003,0.0,0.0,0.0,0.0,0.0,12.5,0.0,0.0,18.0,9.0,...,17.5,11.40,1.0,6.0,7.5,4.0,6.0,0.0,14.00,4.0
2023_01001003_08,2023_01001003,0.0,0.0,0.0,2.0,1.0,11.0,1.0,0.5,7.0,1.0,...,8.0,0.00,1.0,0.0,5.0,1.0,3.5,0.5,18.62,1.0
2023_01001008_03,2023_01001008,0.0,0.0,0.0,0.0,0.0,1.5,0.0,0.0,8.0,0.0,...,4.0,0.50,4.0,0.0,20.0,7.0,4.0,18.5,10.50,0.0


In [37]:
wide_freq_dia.columns

Index(['Agreagado a la leche: Azúcar | Porción: 1 cucharada cafetera copeteada (10g)',
       'Agreagado a la leche: Chocolate u otro saborizante | Porción: 1 cucharada cafetera copeteada (10g)',
       'Agregado a la leche: Azúcar | Porción: 1 cucharada cafetera copeteada (10g)',
       'Agregado a la leche: Chocolate u otro saborizante | Porción: 1 cucharada cafetera copeteada (10g)',
       'Agua sola | Porción: 1 vaso (240 ml)',
       'Agua sola | Porción: 1/2 vaso (120 ml)',
       'Aguacate | Porción: 1 rebanada o 1 pieza de criollo chico (33 g)',
       'Aguas de fruta natural con azúcar | Porción: 1 vaso (240 ml)',
       'Aguas de fruta natural con azúcar | Porción: 1/2 vaso (120 ml)',
       'Aguas de fruta natural sin azúcar | Porción: 1 vaso (240 ml)',
       ...
       'Yogur en vaso: Entero con frutas | Porción: 1 vaso típico de yogur (150 g)',
       'Yogur en vaso: Entero natural | Porción: 1 vaso típico de yogur (150g)',
       'Yogur para beber: Entero con frutas | P

In [36]:
wide_freq_dia #SALVAR ESTA MATRIZ

,descripcion,Agreagado a la leche: Azúcar | Porción: 1 cucharada cafetera copeteada (10g),Agreagado a la leche: Chocolate u otro saborizante | Porción: 1 cucharada cafetera copeteada (10g),Agregado a la leche: Azúcar | Porción: 1 cucharada cafetera copeteada (10g),Agregado a la leche: Chocolate u otro saborizante | Porción: 1 cucharada cafetera copeteada (10g),Agua sola | Porción: 1 vaso (240 ml),Agua sola | Porción: 1/2 vaso (120 ml),Aguacate | Porción: 1 rebanada o 1 pieza de criollo chico (33 g),Aguas de fruta natural con azúcar | Porción: 1 vaso (240 ml),Aguas de fruta natural con azúcar | Porción: 1/2 vaso (120 ml),Aguas de fruta natural sin azúcar | Porción: 1 vaso (240 ml),...,Yogur en vaso: Entero con frutas | Porción: 1 vaso típico de yogur (150 g),Yogur en vaso: Entero natural | Porción: 1 vaso típico de yogur (150g),Yogur para beber: Entero con frutas | Porción: 1 envase típico (230 g),Yogur para beber: Entero natural | Porción: 1 envase típico (230 g),Yogur para beber: a) Entero natural | Porción: 1 envase típico (230 g),Yogur para beber: b) Entero con fruta | Porción: 1 envase típico (230 g),"Yogur para beber: c) Bajo en grasa o light natural o con fruta (vitalinea, activia 0%, Siluette) | Porción: 1 envase típico (230 g)",Zanahoria | Porción: 1 pieza chica o ½ taza (50g),Zanahoria | Porción: 1 pieza chica o ½ taza (50g),Zanahoria | Porción: 1 pieza mediana o ½ taza (50g)
FOLIO_INT,FOLIO_I,,,,,,,,,,,,,,,,,,,,,
2023_01001001_04,2023_01001001,0.0,0.0,0.0,0.0,16.0,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023_01001002_01,2023_01001002,0.0,0.0,0.0,0.0,28.0,0.00,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2023_01001003_06,2023_01001003,0.0,0.0,0.0,0.0,14.0,0.00,2.0,3.5,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023_01001003_08,2023_01001003,0.0,0.0,0.0,0.0,0.0,18.62,10.5,0.0,7.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023_01001008_03,2023_01001008,0.0,0.0,0.0,0.0,10.5,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023_32024023_01,2023_32024023,0.0,0.0,0.0,0.0,28.0,0.00,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
2023_32024024_01,2023_32024024,0.0,0.0,0.0,0.0,35.0,0.00,4.0,28.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2023_32024028_01,2023_32024028,0.0,0.0,0.0,0.0,56.0,0.00,0.0,28.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0


In [42]:
# 1. Definimos la lista de columnas de alimentos (usando las columnas de tu DF)
columnas_alimentos = wide_freq_dia.columns.tolist()

# 2. Creamos la nueva columna 'frecuencia_alimentos'
# Esto generará un diccionario {alimento: valor} para cada fila

dic_alimentos =  {"alimentos por porciones por semana":columnas_alimentos}
dic_alimentos

{'alimentos por porciones por semana': ['Agreagado a la leche: Azúcar | Porción: 1 cucharada cafetera copeteada (10g)',
  'Agreagado a la leche: Chocolate u otro saborizante | Porción: 1 cucharada cafetera copeteada (10g)',
  'Agregado a la leche: Azúcar | Porción: 1 cucharada cafetera copeteada (10g)',
  'Agregado a la leche: Chocolate u otro saborizante | Porción: 1 cucharada cafetera copeteada (10g)',
  'Agua sola | Porción: 1 vaso (240 ml)',
  'Agua sola | Porción: 1/2 vaso (120 ml)',
  'Aguacate | Porción: 1 rebanada o 1 pieza de criollo chico (33 g)',
  'Aguas de fruta natural con azúcar | Porción: 1 vaso (240 ml)',
  'Aguas de fruta natural con azúcar | Porción: 1/2 vaso (120 ml)',
  'Aguas de fruta natural sin azúcar | Porción: 1 vaso (240 ml)',
  'Aguas de fruta natural sin azúcar | Porción: 1/2 vaso (120 ml)',
  'Algún marisco (camarón, ostiones, etc.) | Porción: 1 plato (100g)',
  'Algún marisco (camarón, ostiones, etc.) | Porción: 1/2 plato (50g)',
  'Antojitos con res, cer

In [35]:
import shutil
import os

ruta = '/content/sample_data'

if os.path.exists(ruta):
    shutil.rmtree(ruta)
    print(f"Carpeta {ruta} eliminada con éxito.")
else:
    print("La carpeta no existe.")

Carpeta /content/sample_data eliminada con éxito.


In [34]:
import umap
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler

# 1. Preparación y Reducción de Dimensionalidad (3D)
# Escalamos los datos (vital para métricas euclidianas en UMAP)
scaler = StandardScaler()
features_scaled = scaler.fit_transform(wide_grouped.values)

# Entrenamos el modelo UMAP con 3 componentes
reducer_3d = umap.UMAP(
    n_components=3,
    n_neighbors=15,
    min_dist=0.1,
    metric='euclidean',
    random_state=42
)
embedding_3d = reducer_3d.fit_transform(features_scaled)

# Creamos un DataFrame auxiliar para el plot
df_plot = pd.DataFrame(embedding_3d, columns=['u1', 'u2', 'u3'])
df_plot['FOLIO_I'] = wide_grouped.index.get_level_values('FOLIO_I')

# 2. Definición de Variables y Menú
categories = [
    'ALCOHOL', 'ANTOJ_NOFRY', 'AROMATICS', 'BEV_NATURAL_SUGAR', 'BEV_NON_CALORIC',
    'BEV_SSB', 'CEREAL_SWEET', 'DAIRY_SWEET', 'DAIRY_UNSWEET', 'FAST_FOOD',
    'FATS_ADDED', 'FRIED', 'FRUIT_FRESH', 'FRUIT_PROC', 'INFANT', 'LEGUMES',
    'NUTS_AVO', 'PROT_PROC', 'PROT_UNPROC', 'SNACK_SALTY', 'SOUPS_HOME',
    'SOUP_INST', 'STARCHES', 'SUGAR_ADDED', 'SWEETS', 'VEG', 'WATER', 'WHOLE_GRAINS'
]

# 3. Construcción del Objeto Figure de Plotly
fig = go.Figure()

# Añadimos la traza inicial (usando WATER por defecto)
default_cat = 'WATER'
fig.add_trace(go.Scatter3d(
    x=df_plot['u1'], y=df_plot['u2'], z=df_plot['u3'],
    mode='markers',
    marker=dict(
        size=2.5,
        color=np.log1p(wide_grouped[default_cat]), # Escala log para mayor contraste
        colorscale='Spectral_r',
        opacity=0.7,
        colorbar=dict(title=f"Log Porciones: {default_cat}", thickness=15)
    ),
    text=df_plot['FOLIO_I'], # Info al pasar el cursor
    hoverinfo='text'
))

# 4. Creación del Menú Desplegable (Dropdown)
dropdown_buttons = []

for cat in categories:
    dropdown_buttons.append(dict(
        method="restyle",
        label=cat,
        args=[{
            "marker.color": [np.log1p(wide_grouped[cat])],
            "marker.colorbar.title": f"Log Porciones: {cat}"
        }]
    ))

# 5. Configuración final del Layout
fig.update_layout(
    title="Exploración 3D de Fenotipos Alimentarios (ENSANUT 2023)",
    scene=dict(
        xaxis_title="UMAP 1",
        yaxis_title="UMAP 2",
        zaxis_title="UMAP 3",
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        bgcolor='rgb(20, 20, 20)' # Fondo oscuro para resaltar clústeres
    ),
    updatemenus=[dict(
        buttons=dropdown_buttons,
        direction="down",
        showactive=True,
        x=0.1,
        y=1.15,
        font=dict(color="#000000") # Texto del menú legible
    )],
    margin=dict(l=0, r=0, b=0, t=50),
    template="plotly_dark"
)

fig.show()

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [46]:
# --- 0) Imports ---
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
import hdbscan

import plotly.express as px
import plotly.graph_objects as go

# ------------------------------------------------------------
# Asumo que YA tienes:
# - wide_grouped: DataFrame (index: FOLIO_I u otros niveles; cols: categories)
# - embedding_3d: np.array shape (n, 3)  -> resultado de UMAP
# - categories: lista de columnas
# ------------------------------------------------------------

# --- 1) Preparamos embedding para clustering (recomendado escalarlo) ---
emb_scaled = StandardScaler().fit_transform(embedding_3d)

n = emb_scaled.shape[0]
# Heurística razonable: 1% del tamaño de la muestra, mínimo 50
min_cluster_size = 20

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=min_cluster_size,
    min_samples=None,                 # si quieres clusters más "conservadores", prueba min_samples = min_cluster_size//2
    metric="euclidean",
    cluster_selection_method="eom"    # "leaf" suele dar más clusters pequeños
)

labels = clusterer.fit_predict(emb_scaled)      # -1 es ruido/outliers
proba  = clusterer.probabilities_               # 0..1, “confianza” de asignación

# --- 2) DataFrame con coords + clúster ---
df_plot = pd.DataFrame(embedding_3d, columns=["u1", "u2", "u3"])
df_plot["FOLIO_I"] = wide_grouped.index.get_level_values("FOLIO_I")
df_plot["cluster"] = labels
df_plot["cluster_prob"] = proba
df_plot["is_noise"] = df_plot["cluster"].eq(-1)

print("Clusters (incluyendo ruido=-1):")
print(df_plot["cluster"].value_counts().sort_index())

# --- 3) PERFILADO por clúster usando variables originales ---
X = wide_grouped[categories].copy()

# Sugerencia: perfilar en log(1+porciones) para estabilidad y comparabilidad
X_log = np.log1p(X)

# % consumidores por categoría (por cluster): proporción con consumo > 0
X_cons = (X > 0).astype(int)

# Estadísticos globales (para enriquecimiento)
global_mean = X_log.mean(axis=0)
global_std  = X_log.std(axis=0).replace(0, np.nan)

# Unimos todo para agrupar
df_all = pd.concat(
    [df_plot[["FOLIO_I", "cluster", "cluster_prob", "is_noise"]].set_index("FOLIO_I"),
     X_log.set_index(wide_grouped.index.get_level_values("FOLIO_I")),
     X_cons.set_index(wide_grouped.index.get_level_values("FOLIO_I")).add_suffix("__cons")],
    axis=1
)

# --- 3a) Tablas resumen por clúster (excluyendo ruido opcionalmente) ---
df_in = df_all[~df_all["is_noise"]].copy()

cluster_sizes = df_in.groupby("cluster").size().rename("n")
cluster_pct = (cluster_sizes / cluster_sizes.sum() * 100).rename("pct")

# Medias/medianas en log-porciones
cluster_mean = df_in.groupby("cluster")[categories].mean()
cluster_median = df_in.groupby("cluster")[categories].median()

# % consumidores
cons_cols = [c + "__cons" for c in categories]
cluster_cons_pct = df_in.groupby("cluster")[cons_cols].mean() * 100
cluster_cons_pct.columns = categories  # renombramos para que quede bonito

# Enriquecimiento (z-score): qué tan arriba/abajo del promedio global está cada categoría en cada cluster
cluster_z = (cluster_mean - global_mean) / global_std

summary = pd.concat([cluster_sizes, cluster_pct], axis=1).sort_values("n", ascending=False)
print("\nTamaño de clusters (sin ruido):")
print(summary)

# --- 3b) Función para "leer" perfiles: top categorías enriquecidas y empobrecidas ---
def describe_cluster(k, top=6):
    z = cluster_z.loc[k].sort_values(ascending=False)
    top_pos = z.head(top)
    top_neg = z.tail(top).sort_values()  # más negativos

    out = {
        "cluster": k,
        "n": int(summary.loc[k, "n"]),
        "pct": float(summary.loc[k, "pct"]),
        "top_enriched": list(top_pos.index),
        "top_enriched_z": list(np.round(top_pos.values, 2)),
        "top_depleted": list(top_neg.index),
        "top_depleted_z": list(np.round(top_neg.values, 2)),
    }
    return out

profiles = [describe_cluster(k, top=6) for k in summary.index]
profiles_df = pd.DataFrame(profiles)
print("\nPerfiles (resumen):")
print(profiles_df)

# --- 3c) Perfil completo “largo” por si quieres exportar o graficar fácil ---
cluster_profile_long = (
    cluster_z.reset_index()
    .melt(id_vars="cluster", var_name="category", value_name="z_enrichment")
    .merge(summary.reset_index(), on="cluster", how="left")
)

# ------------------------------------------------------------
# 4) VISUALIZACIONES
# ------------------------------------------------------------

# 4a) UMAP 3D coloreado por cluster (ruido en gris)
df_vis = df_plot.copy()
df_vis["cluster_str"] = df_vis["cluster"].astype(str)
df_vis.loc[df_vis["cluster"].eq(-1), "cluster_str"] = "noise(-1)"

fig_clusters = px.scatter_3d(
    df_vis,
    x="u1", y="u2", z="u3",
    color="cluster_str",
    hover_data=["FOLIO_I", "cluster_prob"],
    opacity=0.75
)
fig_clusters.update_traces(marker=dict(size=2.5))
fig_clusters.update_layout(
    title=f"UMAP 3D + HDBSCAN (min_cluster_size={min_cluster_size})",
    template="plotly_dark",
    margin=dict(l=0, r=0, b=0, t=50),
    scene=dict(
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        zaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        bgcolor='rgb(20, 20, 20)'
    )
)
fig_clusters.show()

# 4b) Heatmap de enriquecimiento (z) por cluster (ordenado por tamaño)
#     (muestra la “firma” del cluster vs promedio global)
order = summary.index.tolist()
z_mat = cluster_z.loc[order, categories]

fig_heat = go.Figure(data=go.Heatmap(
    z=z_mat.values,
    x=categories,
    y=[f"cl{c} (n={summary.loc[c,'n']})" for c in order],
    colorbar=dict(title="z enrichment")
))
fig_heat.update_layout(
    title="Perfiles por clúster (z-score sobre log1p porciones)",
    template="plotly_dark",
    height=350 + 18*len(order),
    margin=dict(l=0, r=0, b=0, t=50),
)
fig_heat.show()

# 4c) (Opcional) barras para un cluster específico: medias y % consumidores
def plot_cluster_bars(k, top=10):
    # Top categorías por enriquecimiento
    z = cluster_z.loc[k].sort_values(ascending=False).head(top)
    cats = z.index.tolist()

    means = cluster_mean.loc[k, cats].values
    cons  = cluster_cons_pct.loc[k, cats].values

    fig = go.Figure()
    fig.add_trace(go.Bar(name="mean log1p(porciones)", x=cats, y=means))
    fig.add_trace(go.Bar(name="% consumidores", x=cats, y=cons))
    fig.update_layout(
        barmode="group",
        title=f"Cluster {k}: top {top} categorías (enriquecidas)",
        template="plotly_dark",
        margin=dict(l=0, r=0, b=0, t=50),
    )
    fig.show()

# Ejemplo:
plot_cluster_bars(summary.index[0], top=12)


Clusters (incluyendo ruido=-1):
cluster
-1     1377
 0       74
 1      127
 2       63
 3       41
 4       51
 5       33
 6       74
 7       82
 8      184
 9       52
 10      43
 11      67
 12      83
 13      65
 14     153
 15      41
 16      70
 17     114
 18      38
 19      91
 20      30
 21      38
 22     172
 23      60
 24     133
Name: count, dtype: int64

Tamaño de clusters (sin ruido):
           n       pct
cluster               
8        184  9.297625
22       172  8.691258
14       153  7.731177
24       133  6.720566
1        127  6.417383
17       114  5.760485
19        91  4.598282
12        83  4.194037
7         82  4.143507
6         74  3.739262
0         74  3.739262
16        70  3.537140
11        67  3.385548
13        65  3.284487
2         63  3.183426
23        60  3.031834
9         52  2.627590
4         51  2.577059
10        43  2.172815
15        41  2.071753
3         41  2.071753
21        38  1.920162
18        38  1.920162
5         33  

#Tortillas

In [ ]:
import pandas as pd
import pyreadstat
import numpy as np
from pathlib import Path
import zipfile

# ==========================================
# CONFIGURACIÓN
# ==========================================
INPUT_ZIP = "/content/nutricion_frecuencia.zip"  # Tu archivo zip
OUT_DIR = Path("/content/ffq_especiales")        # Carpeta para estos archivos
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Directorio temporal de extracción
TMP_DIR = Path("/content/_tmp_special_extract")
TMP_DIR.mkdir(parents=True, exist_ok=True)

# Descomprimir si es necesario
if not any(TMP_DIR.iterdir()):
    print(f"Descomprimiendo {INPUT_ZIP}...")
    with zipfile.ZipFile(INPUT_ZIP, "r") as z:
        z.extractall(TMP_DIR)

# ==========================================
# 1. PROCESAR ARCHIVOS GENERALES (CABECERAS)
#    Objetivo: Extraer p_t_maiz y p_t_trigo
# ==========================================
print("\n--- 1. EXTRACCIÓN DE PESOS DE TORTILLA (CABECERAS) ---")

# Buscamos archivos que NO sean de detalle (_rec, _tor, _sup)
# Típicamente: frec_adul_w.sav, frec_esc_w.sav, frec_pree_w.sav (o nombres similares en la raíz)
# Ajuste: Buscamos archivos .sav que NO tengan guiones bajos intermedios de módulos
all_savs = sorted(list(TMP_DIR.rglob("*.sav")))
header_candidates = [
    f for f in all_savs
    if "_rec_" not in f.name
    and "_tor_" not in f.name
    and "_sup_" not in f.name
    and "actividad" not in f.name
]

weights_list = []

for fp in header_candidates:
    try:
        # Leemos solo metadatos primero para ver si tiene las variables
        _, meta = pyreadstat.read_sav(str(fp), metadataonly=True)
        cols = [c.lower() for c in meta.column_names]

        # Verificamos si tiene las variables de peso de tortilla
        has_maiz = "p_t_maiz" in cols
        has_trigo = "p_t_trigo" in cols

        if has_maiz or has_trigo:
            # Leemos solo las columnas necesarias
            usecols = ["FOLIO_INT"]
            if "folio_i" in cols: usecols.append("FOLIO_I") # A veces cambia
            if has_maiz: usecols.append("p_t_maiz")
            if has_trigo: usecols.append("p_t_trigo")

            # Mapeo para leer mayúsculas/minúsculas correctamente
            real_cols = [c for c in meta.column_names if c.lower() in [u.lower() for u in usecols]]

            df, _ = pyreadstat.read_sav(str(fp), usecols=real_cols)

            # Estandarizar nombres
            df.columns = df.columns.str.upper() # FOLIO_INT en mayúsculas
            if "P_T_MAIZ" in df.columns: df.rename(columns={"P_T_MAIZ": "peso_tortilla_maiz"}, inplace=True)
            if "P_T_TRIGO" in df.columns: df.rename(columns={"P_T_TRIGO": "peso_tortilla_trigo"}, inplace=True)

            # Agregar fuente para rastreo
            df["origen_peso"] = fp.name

            weights_list.append(df)
            print(f"✅ Pesos encontrados en: {fp.name}")
        else:
            # print(f"ℹ️ {fp.name} no tiene datos de peso de tortilla.")
            pass

    except Exception as e:
        print(f"Error leyendo cabecera {fp.name}: {e}")

# Consolidar tabla de pesos
if weights_list:
    df_weights = pd.concat(weights_list, ignore_index=True)
    # Limpieza: Asegurar que sean numéricos y llenar NaNs con promedio o estándar (opcional, aquí dejamos NaN)
    print(f"-> Total sujetos con datos de peso de tortilla: {len(df_weights)}")
    df_weights.to_csv(OUT_DIR / "tabla_pesos_tortilla_por_folio.csv", index=False)
else:
    print("⚠️ No se encontraron variables de peso de tortilla en ningún archivo.")
    df_weights = pd.DataFrame(columns=["FOLIO_INT", "peso_tortilla_maiz", "peso_tortilla_trigo"])

# ==========================================
# 2. PROCESAR MÓDULO TORTILLA (_tor_)
# ==========================================
print("\n--- 2. PROCESAMIENTO DE TORTILLAS ---")
tor_files = sorted(list(TMP_DIR.rglob("*_tor_*.sav")))
dfs_tor = []

for fp in tor_files:
    try:
        df, _ = pyreadstat.read_sav(str(fp))

        # Renombrar columnas específicas de este módulo
        col_map = {
            "folio_int": "FOLIO_INT", "FOLIO_INT": "FOLIO_INT",
            "tortillas": "tipo_tortilla",      # "Tortilla de maíz" / "Tortilla de harina"
            "pa1_t": "dias_semana",            # Días
            "pa4_t": "piezas_por_dia_consumo"  # Piezas (cuando consume)
        }

        # Aplicar renombre (case insensitive)
        actual_cols = {c: col_map[c.lower()] for c in df.columns if c.lower() in col_map}
        df = df.rename(columns=actual_cols)

        # Filtrar columnas clave
        keep = ["FOLIO_INT", "tipo_tortilla", "dias_semana", "piezas_por_dia_consumo"]
        keep = [c for c in keep if c in df.columns]
        df = df[keep].copy()

        df["archivo_origen"] = fp.name

        # Cálculos básicos
        df["dias_semana"] = pd.to_numeric(df["dias_semana"], errors='coerce').fillna(0)
        df["piezas_por_dia_consumo"] = pd.to_numeric(df["piezas_por_dia_consumo"], errors='coerce').fillna(0)

        # Piezas promedio diario = (Días * Piezas_cuando_come) / 7
        df["piezas_diarias_promedio"] = (df["dias_semana"] * df["piezas_por_dia_consumo"]) / 7.0

        dfs_tor.append(df)
        print(f"✅ Tortillas: {fp.name}")

    except Exception as e:
        print(f"Error en tortilla {fp.name}: {e}")

if dfs_tor:
    df_tortilla = pd.concat(dfs_tor, ignore_index=True)

    # --- CRUCE MÁGICO: UNIR CON LOS PESOS ---
    # Unimos con la tabla de pesos usando FOLIO_INT
    # Primero preparamos df_weights para que tenga una sola fila por folio (si hubiera duplicados)
    df_w_unique = df_weights.groupby("FOLIO_INT")[["peso_tortilla_maiz", "peso_tortilla_trigo"]].first().reset_index()

    df_tortilla = df_tortilla.merge(df_w_unique, on="FOLIO_INT", how="left")

    # Calcular GRAMOS
    # Lógica: Si el tipo es "maíz", usa peso_maiz; si es "harina/trigo", usa peso_trigo

    # Normalizar texto para búsqueda
    df_tortilla["tipo_norm"] = df_tortilla["tipo_tortilla"].astype(str).str.lower()

    conditions = [
        df_tortilla["tipo_norm"].str.contains("maiz|maíz", na=False),
        df_tortilla["tipo_norm"].str.contains("harina|trigo", na=False)
    ]
    choices = [
        df_tortilla["piezas_diarias_promedio"] * df_tortilla["peso_tortilla_maiz"],
        df_tortilla["piezas_diarias_promedio"] * df_tortilla["peso_tortilla_trigo"]
    ]

    # Gramos diarios (si no encuentra peso, queda NaN)
    df_tortilla["gramos_diarios_promedio"] = np.select(conditions, choices, default=np.nan)

    # Limpieza final
    df_tortilla.drop(columns=["tipo_norm"], inplace=True)

    file_tor = OUT_DIR / "tortillas_consolidado_con_gramos.csv"
    df_tortilla.to_csv(file_tor, index=False)
    print(f"💾 Guardado: {file_tor}")
    print(df_tortilla[["FOLIO_INT", "tipo_tortilla", "piezas_diarias_promedio", "gramos_diarios_promedio"]].head())

# ==========================================
# 3. PROCESAR MÓDULO SUPLEMENTOS (_sup_)
# ==========================================
print("\n--- 3. PROCESAMIENTO DE SUPLEMENTOS ---")
sup_files = sorted(list(TMP_DIR.rglob("*_sup_*.sav")))
dfs_sup = []

for fp in sup_files:
    try:
        df, _ = pyreadstat.read_sav(str(fp))

        # Renombrar
        col_map = {
            "folio_int": "FOLIO_INT", "FOLIO_INT": "FOLIO_INT",
            "suplemento": "nombre_suplemento",
            "pa1_s": "dias_semana",
            "pa2_s": "veces_al_dia",
            "pa4_s": "cantidad_porciones",
            "esp_otro": "especifique_otro"
        }

        actual_cols = {c: col_map[c.lower()] for c in df.columns if c.lower() in col_map}
        df = df.rename(columns=actual_cols)

        # En suplementos a veces 'suplemento' es un código y el nombre está en otra parte o etiqueta.
        # Asumiremos que pyreadstat trae las etiquetas si no usamos apply_value_formats=False.
        # Aquí leemos normal, así que 'nombre_suplemento' debería tener el texto si es variable categórica.

        cols_final = ["FOLIO_INT", "nombre_suplemento", "especifique_otro", "dias_semana", "veces_al_dia", "cantidad_porciones"]
        cols_final = [c for c in cols_final if c in df.columns]

        df = df[cols_final].copy()
        df["archivo_origen"] = fp.name

        # Filtro de consumo (si dias_semana > 0)
        df["dias_semana"] = pd.to_numeric(df["dias_semana"], errors='coerce').fillna(0)
        df = df[df["dias_semana"] > 0]

        dfs_sup.append(df)
        print(f"✅ Suplementos: {fp.name}")

    except Exception as e:
        print(f"Error en suplementos {fp.name}: {e}")

if dfs_sup:
    df_suplementos = pd.concat(dfs_sup, ignore_index=True)
    file_sup = OUT_DIR / "suplementos_consolidado.csv"
    df_suplementos.to_csv(file_sup, index=False)
    print(f"💾 Guardado: {file_sup}")
    print(df_suplementos.head())

print("\n--- PROCESO TERMINADO ---")
print(f"Archivos especiales generados en: {OUT_DIR}")

In [ ]:

df_suplementos